# TP3 — Paso 1: cargar y revisar los libros

## Paso 1 Cargar y revisar los libros

Se conserva slug como identificador, title para mostrar el libro, authors y synopsis como datos de entrada. Solo se normalizan los espacios; se mantienen los demás caracteres porque pueden ser evidencia en la auditoría del Paso 2.

In [1]:
%pip install -q python-Levenshtein

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import unicodedata
from collections import defaultdict
from itertools import combinations

import Levenshtein

def normalizar_espacios(texto):
    return re.sub(r"\s+", " ", str(texto or "")).strip()

def normalizar_titulo(texto):
    texto = unicodedata.normalize("NFKC", str(texto or "")).casefold()
    texto = re.sub(r"[^\w\s]", " ", texto)
    return normalizar_espacios(texto)

def jaccard_palabras(texto_a, texto_b):
    palabras_a = set(re.findall(r"\w+", str(texto_a or "").casefold()))
    palabras_b = set(re.findall(r"\w+", str(texto_b or "").casefold()))
    union = palabras_a | palabras_b
    if not union:
        return 0.0
    return len(palabras_a & palabras_b) / len(union)

def limpiar_catalogo(datos):
    catalogo = datos.copy()
    catalogo["synopsis"] = catalogo["synopsis"].fillna("").apply(normalizar_espacios)
    conteos = {"sinopsis_vacia": 0, "slug_duplicado": 0, "sinopsis_duplicada": 0}

    vacias = catalogo["synopsis"].eq("")
    conteos["sinopsis_vacia"] = int(vacias.sum())
    catalogo = catalogo.loc[~vacias].copy()

    duplicados_slug = catalogo["slug"].duplicated()
    conteos["slug_duplicado"] = int(duplicados_slug.sum())
    catalogo = catalogo.loc[~duplicados_slug].copy()

    duplicadas = catalogo["synopsis"].duplicated()
    conteos["sinopsis_duplicada"] = int(duplicadas.sum())
    catalogo = catalogo.loc[~duplicadas].copy()

    return catalogo.reset_index(drop=True), conteos

UMBRAL_TITULO = 0.80
UMBRAL_SINOPSIS = 0.75
AUTORES_GENERICOS = {"varios autores", "anónimo", "anonimo"}

def generar_candidatos(datos):
    filas_por_autor = defaultdict(set)
    nombre_autor = {}
    for indice, autores in datos["authors"].items():
        if isinstance(autores, str):
            autores = [autores]
        elif not isinstance(autores, (list, tuple, set)):
            autores = []
        for autor in autores:
            autor = normalizar_espacios(autor)
            clave = autor.casefold()
            if clave and clave not in AUTORES_GENERICOS:
                filas_por_autor[clave].add(indice)
                nombre_autor[clave] = autor

    autores_compartidos = defaultdict(set)
    for clave, indices in filas_por_autor.items():
        for indice_a, indice_b in combinations(sorted(indices), 2):
            autores_compartidos[(indice_a, indice_b)].add(nombre_autor[clave])

    resultados = []
    for (indice_a, indice_b), autores in autores_compartidos.items():
        libro_a = datos.loc[indice_a]
        libro_b = datos.loc[indice_b]
        similitud_titulo = Levenshtein.ratio(
            normalizar_titulo(libro_a["title"]),
            normalizar_titulo(libro_b["title"]),
        )
        if similitud_titulo < UMBRAL_TITULO:
            continue
        similitud_sinopsis = jaccard_palabras(libro_a["synopsis"], libro_b["synopsis"])
        if similitud_sinopsis < UMBRAL_SINOPSIS:
            continue
        resultados.append({
            "slug_a": libro_a["slug"],
            "titulo_a": libro_a["title"],
            "slug_b": libro_b["slug"],
            "titulo_b": libro_b["title"],
            "autores_compartidos": ", ".join(sorted(autores)),
            "similitud_titulo": round(similitud_titulo, 3),
            "jaccard_sinopsis": round(similitud_sinopsis, 3),
        })

    columnas = ["slug_a", "titulo_a", "slug_b", "titulo_b", "autores_compartidos", "similitud_titulo", "jaccard_sinopsis"]
    if not resultados:
        return pd.DataFrame(columns=columnas)
    return pd.DataFrame(resultados, columns=columnas).sort_values(
        ["jaccard_sinopsis", "similitud_titulo"], ascending=False
    ).reset_index(drop=True)

### Cargar el catálogo

Los archivos de entrada están versionados en la carpeta data del repositorio. La celda los busca al ejecutar Jupyter desde la raíz o desde notebooks. En Colab, se pueden cargar los archivos en data o en el directorio actual.

In [3]:
from pathlib import Path
import pandas as pd
from IPython.display import display

rutas_datos = [Path("data"), Path("../data"), Path("../../TP3"), Path("../../../TP3"), Path(".")]
directorio_datos = next(
    (ruta for ruta in rutas_datos if (ruta / "catalogo_tp3.jsonl").is_file()),
    None,
)
if directorio_datos is None:
    raise FileNotFoundError("No se encontró catalogo_tp3.jsonl en las rutas de datos probadas.")

catalogo_original = pd.read_json(directorio_datos / "catalogo_tp3.jsonl", lines=True)
columnas_paso1 = ["slug", "title", "authors", "synopsis"]
catalogo = catalogo_original[columnas_paso1].copy()
print(f"Registros originales: {len(catalogo):,}")

Registros originales: 8,747


### Revisar sinopsis repetidas

Primero se normalizan solo los espacios y se muestran los grupos con la misma sinopsis antes de quitar duplicados. Para esta práctica se conserva el primer registro de cada grupo, tal como indica el enunciado.

In [4]:
catalogo_revision = catalogo.copy()
catalogo_revision["synopsis"] = catalogo_revision["synopsis"].fillna("").apply(normalizar_espacios)
con_sinopsis = catalogo_revision.loc[catalogo_revision["synopsis"].ne("")]
duplicadas = con_sinopsis.loc[con_sinopsis["synopsis"].duplicated(keep=False)]

grupos_sinopsis = (
    duplicadas.groupby("synopsis", as_index=False)
    .agg(
        cantidad=("slug", "size"),
        titulos=("title", lambda titulos: " / ".join(titulos.iloc[:5]) + (" / …" if len(titulos) > 5 else "")),
    )
)
grupos_sinopsis["fragmento"] = grupos_sinopsis["synopsis"].str.slice(0, 140)
filas_repetidas = int((grupos_sinopsis["cantidad"] - 1).sum())
print(f"Grupos con sinopsis repetida: {len(grupos_sinopsis)}")
print(f"Registros que se quitarían por este motivo: {filas_repetidas}")
display(grupos_sinopsis.drop(columns="synopsis").sort_values("cantidad", ascending=False).reset_index(drop=True))

Grupos con sinopsis repetida: 33
Registros que se quitarían por este motivo: 98


,cantidad,titulos,fragmento
0,58,Alemán – Unidad 40 / Alemán – Unidad 15 / Alem...,El Curso de Alemán consta de 96 Unidades didác...
1,7,Nueva Dimensión 7 / Nueva Dimensión 5 / Nueva ...,Revista española de ciencia ficción y fantasía...
2,4,Teatro crítico universal VII / Teatro crítico ...,El «Teatro crítico universal» (teatro ha de en...
3,3,Dionisíacas Cantos XXV-XXXVI / Dionisíacas Can...,"Nono (siglo V d. C.), natural de Panópolis, en..."
4,3,El fin de una cuadrilla / El santón de la mont...,"El autor, llamado Kara Ben Nemsi (Carlos, hijo..."
5,2,Prácticas sexuales alternativas (I) / Práctica...,Al repasar la larga lista de prácticas sexuale...
6,2,Los años irreverentes. Escritos completos en E...,"Como todas las cosas que suceden en la vida, h..."
7,2,Comunicaciones unificadas con Elastix / Comuni...,Comunicaciones Unificadas con Elastix es una o...
8,2,La Historia Interminable – Color / La Historia...,"Bastián Baltasar Bux, es un niño tímido al que..."
9,2,El 19 de marzo y el 2 de mayo / Bailén,El gran friso narrativo de los Episodios Nacio...


### Limpiar registros y conservar una sinopsis por grupo

In [5]:
catalogo_limpio, conteos_eliminacion = limpiar_catalogo(catalogo)

resumen_limpieza = pd.DataFrame(
    [{"motivo": motivo, "eliminados": cantidad} for motivo, cantidad in conteos_eliminacion.items()]
)
resumen_limpieza.loc[len(resumen_limpieza)] = ["disponibles tras limpieza exacta", len(catalogo_limpio)]
display(resumen_limpieza)
print(f"Libros disponibles tras eliminar sinopsis vacías y repetidas: {len(catalogo_limpio):,}")


,motivo,eliminados
0,sinopsis_vacia,1
1,slug_duplicado,0
2,sinopsis_duplicada,98
3,disponibles tras limpieza exacta,8648


Libros disponibles tras eliminar sinopsis vacías y repetidas: 8,648


### Revisar títulos y sinopsis parecidos

Los pares se numeran para revisar cada caso. Los umbrales solo proponen candidatos; no determinan fusiones.


In [6]:
candidatos = generar_candidatos(catalogo_limpio)
candidatos.insert(0, "id_candidato", range(1, len(candidatos) + 1))
print(f"Pares que superan ambos filtros: {len(candidatos)}")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None, "display.width", 220):
    display(candidatos)


Pares que superan ambos filtros: 66


,id_candidato,slug_a,titulo_a,slug_b,titulo_b,autores_compartidos,similitud_titulo,jaccard_sinopsis
0,1,comentarios-psicologicos-sobre-las-ensenanzas-de-gurdjieff-y-ouspensky-2,Comentarios psicológicos sobre las enseñanzas de Gurdjieff y Ouspensky,comentarios-psicologicos-sobre-las-ensenanzas-de-gurdjieff-y-ouspensky,Comentarios psicológicos sobre las enseñanzas de Gurdjieff y Ouspensky,Maurice Nicoll,1.000,1.000
1,2,vidas-magicas-e-inquisicion-i,Vidas mágicas e Inquisición I,vidas-magicas-e-inquisicion-ii,Vidas mágicas e Inquisición II,Julio Caro Baroja,0.983,1.000
2,3,orgullo-y-poco-juicio-parte-2,Orgullo y poco Juicio. Parte 2,orgullo-y-poco-juicio-parte-1,Orgullo y poco Juicio. Parte 1,Lena Valenti,0.966,1.000
3,4,teatro-critico-universal-vii,Teatro crítico universal VII,teatro-critico-universal-i,Teatro crítico universal I,Benito Jerónimo Feijoo,0.963,1.000
4,5,epigramas-ii,Epigramas II,epigramas-i,Epigramas I,Marco Valerio Marcial,0.957,1.000
5,6,cartas-eruditas-y-curiosas-ii,"Cartas eruditas, y curiosas II",cartas-eruditas-y-curiosas-v,"Cartas eruditas, y curiosas V",Benito Jerónimo Feijoo,0.947,1.000
6,7,manual-para-viajeros-por-andalucia-y-lectores-en-casa-granada,Manual para viajeros por Andalucía y lectores en casa. Granada,manual-para-viajeros-por-andalucia-y-lectores-en-casa,Manual para viajeros por Andalucía y lectores en casa,Richard Ford,0.930,1.000
7,8,ella-hija-de-la-sabiduria,"Ella, Hija de la Sabiduría",ella-hija-de-la-sabiduria-h-rider-haggard,Ella. Hija de la sabiduría,H. Rider Haggard,1.000,0.989
8,9,teatro-critico-universal-vii,Teatro crítico universal VII,teatro-critico-universal-ii,Teatro crítico universal II,Benito Jerónimo Feijoo,0.982,0.987
9,10,teatro-critico-universal-i,Teatro crítico universal I,teatro-critico-universal-ii,Teatro crítico universal II,Benito Jerónimo Feijoo,0.981,0.987


### Decisiones manuales y catálogo final

Tras revisar los candidatos, se aprobaron las fusiones #1, #7, #8, #11 y #31. En cada par se conserva la primera ficha según el orden del catálogo y se quita la segunda. Las razones aparecen junto con los títulos, el autor y los puntajes.

El candidato #7, «Manual para viajeros por Andalucía y lectores en casa. Granada» / «Manual para viajeros por Andalucía y lectores en casa», también se une. Comparten autor y su Jaccard de sinopsis es 1,00: tienen el mismo conjunto de palabras al convertirlas a minúsculas, aunque las sinopsis no sean necesariamente idénticas carácter por carácter. Se conserva la primera ficha del catálogo y se elimina la segunda, aunque su título no especifique Granada.

Los otros dos ejemplos de separación son «Conversaciones con Dios I/II» y «La escultura del siglo XIX/XX»: se parecen en el título, pero describen volúmenes o contenidos diferentes.


In [7]:
motivos_fusion = {
    1: "Mismo autor y título idéntico; las sinopsis difieren solo en detalles de puntuación.",
    7: "Mismo autor y el mismo conjunto de palabras en la sinopsis (Jaccard = 1,00); se conserva la primera ficha aunque la otra especifique Granada.",
    8: "Mismo autor y obra; el título cambia puntuación y mayúsculas, y las sinopsis son casi iguales.",
    11: "Mismo autor y obra; una ficha omite el artículo inicial «El» y las sinopsis casi coinciden.",
    31: "Mismo autor y obra; una ficha omite el artículo inicial «El» y las sinopsis describen el mismo libro.",
}
ids_fusion_aprobados = list(motivos_fusion)
fusiones_aprobadas = candidatos.loc[candidatos["id_candidato"].isin(ids_fusion_aprobados)].copy()
fusiones_aprobadas["decision"] = "unir"
fusiones_aprobadas["motivo"] = fusiones_aprobadas["id_candidato"].map(motivos_fusion)
fusiones_aprobadas["slug_conservado"] = fusiones_aprobadas["slug_a"]
fusiones_aprobadas["slug_eliminado"] = fusiones_aprobadas["slug_b"]
with pd.option_context("display.max_colwidth", None, "display.width", 220):
    display(fusiones_aprobadas[["id_candidato", "titulo_a", "titulo_b", "autores_compartidos", "similitud_titulo", "jaccard_sinopsis", "decision", "motivo", "slug_conservado", "slug_eliminado"]])

slugs_a_eliminar = fusiones_aprobadas["slug_eliminado"].tolist()
catalogo_final = catalogo_limpio.loc[~catalogo_limpio["slug"].isin(slugs_a_eliminar)].reset_index(drop=True)

pares_separados = [
    ("conversaciones-con-dios-i", "conversaciones-con-dios-ii", "Son volúmenes distintos; sus sinopsis describen contenidos diferentes."),
    ("la-escultura-del-siglo-xix", "la-escultura-del-siglo-xx", "Se refieren a siglos y contenidos distintos."),
]
separaciones = []
for slug_a, slug_b, motivo in pares_separados:
    libro_a = catalogo_limpio.loc[catalogo_limpio["slug"].eq(slug_a)].iloc[0]
    libro_b = catalogo_limpio.loc[catalogo_limpio["slug"].eq(slug_b)].iloc[0]
    separaciones.append({
        "titulo_a": libro_a["title"],
        "titulo_b": libro_b["title"],
        "autor": ", ".join(sorted(set(libro_a["authors"]) & set(libro_b["authors"]))),
        "similitud_titulo": round(Levenshtein.ratio(normalizar_titulo(libro_a["title"]), normalizar_titulo(libro_b["title"])), 3),
        "jaccard_sinopsis": round(jaccard_palabras(libro_a["synopsis"], libro_b["synopsis"]), 3),
        "decision": "separar",
        "motivo": motivo,
    })
with pd.option_context("display.max_colwidth", None, "display.width", 220):
    display(pd.DataFrame(separaciones))

resumen_final = pd.DataFrame([
    {"motivo": motivo, "eliminados": cantidad}
    for motivo, cantidad in conteos_eliminacion.items()
])
resumen_final.loc[len(resumen_final)] = ["fusion aproximada aprobada", len(slugs_a_eliminar)]
resumen_final.loc[len(resumen_final)] = ["libros finales", len(catalogo_final)]
display(resumen_final)
print(f"Slugs únicos: {catalogo_final['slug'].nunique():,} de {len(catalogo_final):,}")
print(f"Sinopsis no vacías: {catalogo_final['synopsis'].ne('').sum():,}")
print(f"Sinopsis únicas: {catalogo_final['synopsis'].nunique():,} de {len(catalogo_final):,}")

muestra_final = catalogo_final[["slug", "title", "authors", "synopsis"]].head(5)
display(muestra_final)


,id_candidato,titulo_a,titulo_b,autores_compartidos,similitud_titulo,jaccard_sinopsis,decision,motivo,slug_conservado,slug_eliminado
0,1,Comentarios psicológicos sobre las enseñanzas de Gurdjieff y Ouspensky,Comentarios psicológicos sobre las enseñanzas de Gurdjieff y Ouspensky,Maurice Nicoll,1.000,1.000,unir,Mismo autor y título idéntico; las sinopsis difieren solo en detalles de puntuación.,comentarios-psicologicos-sobre-las-ensenanzas-de-gurdjieff-y-ouspensky-2,comentarios-psicologicos-sobre-las-ensenanzas-de-gurdjieff-y-ouspensky
6,7,Manual para viajeros por Andalucía y lectores en casa. Granada,Manual para viajeros por Andalucía y lectores en casa,Richard Ford,0.930,1.000,unir,"Mismo autor y el mismo conjunto de palabras en la sinopsis (Jaccard = 1,00); se conserva la primera ficha aunque la otra especifique Granada.",manual-para-viajeros-por-andalucia-y-lectores-en-casa-granada,manual-para-viajeros-por-andalucia-y-lectores-en-casa
7,8,"Ella, Hija de la Sabiduría",Ella. Hija de la sabiduría,H. Rider Haggard,1.000,0.989,unir,"Mismo autor y obra; el título cambia puntuación y mayúsculas, y las sinopsis son casi iguales.",ella-hija-de-la-sabiduria,ella-hija-de-la-sabiduria-h-rider-haggard
10,11,El príncipe y el mendigo,Príncipe y mendigo,Mark Twain,0.857,0.985,unir,Mismo autor y obra; una ficha omite el artículo inicial «El» y las sinopsis casi coinciden.,el-principe-y-el-mendigo,principe-y-mendigo
30,31,El contrato social,Contrato social,Jean-Jacques Rousseau,0.909,0.917,unir,Mismo autor y obra; una ficha omite el artículo inicial «El» y las sinopsis describen el mismo libro.,el-contrato-social,contrato-social


,titulo_a,titulo_b,autor,similitud_titulo,jaccard_sinopsis,decision,motivo
0,Conversaciones Con Dios I,Conversaciones Con Dios II,Neale Donald Walsch,0.98,0.094,separar,Son volúmenes distintos; sus sinopsis describen contenidos diferentes.
1,La escultura del siglo XIX,La escultura del siglo XX,Ernesto Ballesteros Arranz,0.98,0.130,separar,Se refieren a siglos y contenidos distintos.


,motivo,eliminados
0,sinopsis_vacia,1
1,slug_duplicado,0
2,sinopsis_duplicada,98
3,fusion aproximada aprobada,5
4,libros finales,8643


Slugs únicos: 8,643 de 8,643
Sinopsis no vacías: 8,643
Sinopsis únicas: 8,643 de 8,643


,slug,title,authors,synopsis
0,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],La Humanidad ha entrado en una nueva fase evol...
1,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],La extracción de ADN de los huesos de nuestros...
2,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],Teotihuacan es la primera gran ciudad del cent...
3,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],Entre los siglos VIII y VI a. C. los fenicios ...
4,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],Las raíces evolutivas del sufrimiento humano U...


## Paso 2 Auditar el catálogo

La muestra etiquetada se divide por libro antes de evaluar los detectores. La partición de desarrollo queda disponible para los subpasos siguientes y la partición de prueba se reserva para medir los resultados.

In [8]:
from sklearn.model_selection import train_test_split

muestra = pd.read_csv(directorio_datos / "muestra_etiquetada.csv", keep_default_na=False)
desarrollo, prueba = train_test_split(
    muestra,
    test_size=0.3,
    random_state=42,
    stratify=muestra["contaminado"],
)

resumen_particion = pd.DataFrame([
    {
        "particion": nombre,
        "libros": len(datos),
        "limpios": int(datos["contaminado"].eq(0).sum()),
        "contaminados": int(datos["contaminado"].eq(1).sum()),
    }
    for nombre, datos in [("desarrollo", desarrollo), ("prueba", prueba)]
])
display(resumen_particion)

,particion,libros,limpios,contaminados
0,desarrollo,350,280,70
1,prueba,150,120,30


### 2.1 Expresiones regulares

Se buscan por separado URLs, correos y teléfonos en la sinopsis original. La combinación marca el libro cuando encuentra al menos una de esas señales. Las formas disimuladas se revisarán en el apartado 2.2.

In [9]:
from sklearn.metrics import precision_score, recall_score

patrones_regex = [
    ("URL", "marca_url", re.compile(r"https?://|www\.", re.IGNORECASE)),
    ("Correo", "marca_correo", re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", re.IGNORECASE)),
    ("Teléfono", "marca_telefono", re.compile(r"\+\d(?:[\s().-]?\d){7,14}")),
]

for nombre, columna, patron in patrones_regex:
    prueba[columna] = prueba["synopsis"].str.contains(patron, na=False)

columnas_individuales = [columna for _, columna, _ in patrones_regex]
prueba["marca_combinada"] = prueba[columnas_individuales].any(axis=1)

columnas_detectores = {nombre: columna for nombre, columna, _ in patrones_regex}
columnas_detectores["Combinada"] = "marca_combinada"
y_prueba = prueba["contaminado"].astype(int)
resultados_regex = []
for nombre, columna in columnas_detectores.items():
    predicciones = prueba[columna].astype(int)
    resultados_regex.append({
        "detector": nombre,
        "precision": round(precision_score(y_prueba, predicciones, zero_division=0), 2),
        "recall": round(recall_score(y_prueba, predicciones, zero_division=0), 2),
        "libros_marcados": int(predicciones.sum()),
    })
tabla_metricas_regex = pd.DataFrame(resultados_regex)
display(tabla_metricas_regex)

falsos_positivos_regex = prueba.loc[
    prueba["contaminado"].eq(0) & prueba["marca_combinada"],
].copy()
falsos_positivos_regex["senales"] = falsos_positivos_regex.apply(
    lambda fila: ", ".join(
        nombre for nombre, columna, _ in patrones_regex if fila[columna]
    ),
    axis=1,
)
print(f"Falsos positivos de la combinación: {len(falsos_positivos_regex)}")
display(falsos_positivos_regex[["slug", "title", "senales", "synopsis"]])

if falsos_positivos_regex.empty:
    desarrollo_marcado = desarrollo.copy()
    for _, columna, patron in patrones_regex:
        desarrollo_marcado[columna] = desarrollo_marcado["synopsis"].str.contains(patron, na=False)
    desarrollo_marcado["marca_combinada"] = desarrollo_marcado[columnas_individuales].any(axis=1)
    falsos_positivos_desarrollo = desarrollo_marcado.loc[
        desarrollo_marcado["contaminado"].eq(0) & desarrollo_marcado["marca_combinada"],
    ].copy()
    falsos_positivos_desarrollo["senales"] = falsos_positivos_desarrollo.apply(
        lambda fila: ", ".join(
            nombre for nombre, columna, _ in patrones_regex if fila[columna]
        ),
        axis=1,
    )
    print("Ejemplos limpios de desarrollo; no forman parte de las métricas de prueba:")
    display(falsos_positivos_desarrollo[["slug", "title", "senales", "synopsis"]])


,detector,precision,recall,libros_marcados
0,URL,1.0,0.13,4
1,Correo,1.0,0.07,2
2,Teléfono,1.0,0.03,1
3,Combinada,1.0,0.23,7


Falsos positivos de la combinación: 0


,slug,title,senales,synopsis


Ejemplos limpios de desarrollo; no forman parte de las métricas de prueba:


,slug,title,senales,synopsis
234,que-pasaria-si,¿Que pasaría si…?,URL,Un libro en el que el humor y la ciencia se un...


### Respuesta a la pregunta del enunciado

**¿Qué tipo de contaminación detecta bien la regex y cuál no?**

Entre las señales buscadas, las URLs explícitas fueron las más detectadas: el patrón de URL marcó 4 de los 30 libros contaminados de prueba (recall global 0,13). El de correo marcó 2 (0,07) y el de teléfono 1 (0,03). La combinación OR marcó 7 de 30 (recall 0,23). Las URLs fueron la señal más útil en esta muestra, aunque el recall de la combinación sigue siendo bajo.

Las etiquetas solo indican si cada libro está contaminado; no identifican el tipo de contaminación. Por eso, esos valores son recall sobre los 30 libros contaminados en conjunto, no recall específico de libros con URL, correo o teléfono. Las regex tampoco detectan contaminación que no incluya esas señales, como enlaces disimulados o texto promocional sin datos de contacto.

La prueba no tuvo falsos positivos. En desarrollo, el libro limpio «¿Qué pasaría si…?» contiene el sitio real `http://www.xkcd.com` y lo marca la regex de URL. Ese caso se muestra aparte y no se usa para calcular métricas. Cada falso positivo puede quitar del catálogo un libro legítimo y dejar de recomendarse.

La muestra está enriquecida: contiene 20 % de libros contaminados, una proporción mayor que la del catálogo. Por eso, su precisión sirve para comparar estos detectores en la muestra, pero no representa directamente la precisión esperable al aplicarlos al catálogo completo.

### 2.2 Normalizar antes de buscar

La regex del apartado 2.1 busca señales explícitas. Algunos enlaces separan sus letras y signos, escriben los signos con palabras o usan caracteres Unicode parecidos a letras latinas. La función `normalizar(texto)` modifica una copia para buscar esas formas; las sinopsis originales se conservan.

Se mantienen los mismos patrones y la misma partición: desarrollo para revisar ejemplos y prueba para comparar las métricas.

In [10]:
homoglifos = str.maketrans({"е": "e", "а": "a"})


def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto or "")).lower()
    texto = "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
        and unicodedata.category(caracter) != "Cf"
    )
    texto = texto.translate(homoglifos)
    texto = re.sub(r"\[\s*\.\s*\]", ".", texto)
    texto = re.sub(r"\bpunto\b", ".", texto)
    texto = re.sub(r"\bbarra\b", "/", texto)
    texto = re.sub(
        r"\bh\s+t\s+t\s+p\s*s?\b",
        lambda coincidencia: re.sub(r"\s+", "", coincidencia.group()),
        texto,
    )
    texto = re.sub(r"\bw\s+w\s+w\b", "www", texto)
    texto = re.sub(r"\s*([:/@])\s*", r"\1", texto)
    texto = re.sub(r"(?<=\w)\s*\.\s*(?=\w)", ".", texto)
    return texto

In [11]:
prueba_2_2 = prueba.copy()
prueba_2_2["synopsis_normalizada"] = prueba_2_2["synopsis"].map(normalizar)

columnas_detectores_normalizados = {}
for nombre, columna, patron in patrones_regex:
    columna_normalizada = f"{columna}_normalizada"
    prueba_2_2[columna_normalizada] = prueba_2_2["synopsis_normalizada"].map(
        lambda texto: bool(patron.search(texto))
    )
    columnas_detectores_normalizados[nombre] = columna_normalizada

columnas_marcadas_normalizadas = list(columnas_detectores_normalizados.values())
prueba_2_2["marca_combinada_normalizada"] = prueba_2_2[
    columnas_marcadas_normalizadas
].any(axis=1)
columnas_detectores_normalizados["Combinada"] = "marca_combinada_normalizada"

y_prueba_2_2 = prueba_2_2["contaminado"].astype(int)
resultados_normalizados = []
for nombre, columna in columnas_detectores_normalizados.items():
    predicciones = prueba_2_2[columna].astype(int)
    resultados_normalizados.append({
        "detector": nombre,
        "precision": round(precision_score(y_prueba_2_2, predicciones, zero_division=0), 2),
        "recall": round(recall_score(y_prueba_2_2, predicciones, zero_division=0), 2),
        "libros_marcados": int(predicciones.sum()),
    })

tabla_metricas_regex_2_2 = pd.concat(
    [
        tabla_metricas_regex.assign(version="Original"),
        pd.DataFrame(resultados_normalizados).assign(version="Normalizada"),
    ],
    ignore_index=True,
)[["version", "detector", "precision", "recall", "libros_marcados"]]
display(tabla_metricas_regex_2_2)

falsos_positivos_normalizados = prueba_2_2.loc[
    prueba_2_2["contaminado"].eq(0)
    & prueba_2_2["marca_combinada_normalizada"]
].copy()
print(f"Falsos positivos de la combinación normalizada: {len(falsos_positivos_normalizados)}")
if falsos_positivos_normalizados.empty:
    print("La partición de prueba no tuvo falsos positivos normalizados.")
else:
    falsos_positivos_normalizados["senales"] = falsos_positivos_normalizados.apply(
        lambda fila: ", ".join(
            nombre
            for nombre, columna in columnas_detectores_normalizados.items()
            if nombre != "Combinada" and fila[columna]
        ),
        axis=1,
    )
    display(
        falsos_positivos_normalizados[
            ["slug", "title", "senales", "synopsis"]
        ]
    )

,version,detector,precision,recall,libros_marcados
0,Original,URL,1.0,0.13,4
1,Original,Correo,1.0,0.07,2
2,Original,Teléfono,1.0,0.03,1
3,Original,Combinada,1.0,0.23,7
4,Normalizada,URL,1.0,0.23,7
5,Normalizada,Correo,1.0,0.07,2
6,Normalizada,Teléfono,1.0,0.03,1
7,Normalizada,Combinada,1.0,0.33,10


Falsos positivos de la combinación normalizada: 0
La partición de prueba no tuvo falsos positivos normalizados.


La tabla compara cada regex y su combinación antes y después de normalizar, siempre sobre los mismos libros de prueba. Los ejemplos siguientes salen de desarrollo y no forman parte de las métricas.

In [12]:
desarrollo_2_2 = desarrollo.copy()
columnas_originales_desarrollo = []
columnas_normalizadas_desarrollo = []
for _, columna, patron in patrones_regex:
    columna_original = f"{columna}_original"
    columna_normalizada = f"{columna}_normalizada"
    desarrollo_2_2[columna_original] = desarrollo_2_2["synopsis"].map(
        lambda texto: bool(patron.search(texto))
    )
    desarrollo_2_2[columna_normalizada] = desarrollo_2_2["synopsis"].map(
        lambda texto: bool(patron.search(normalizar(texto)))
    )
    columnas_originales_desarrollo.append(columna_original)
    columnas_normalizadas_desarrollo.append(columna_normalizada)

desarrollo_2_2["marca_combinada_original"] = desarrollo_2_2[
    columnas_originales_desarrollo
].any(axis=1)
desarrollo_2_2["marca_combinada_normalizada"] = desarrollo_2_2[
    columnas_normalizadas_desarrollo
].any(axis=1)


def senales_fila(fila, sufijo):
    return ", ".join(
        nombre
        for nombre, columna, _ in patrones_regex
        if fila[f"{columna}_{sufijo}"]
    )


slugs_recuperados = [
    "espana-terra-incognita",
    "los-crimenes-del-invisible",
]
ejemplos_recuperados = desarrollo_2_2.loc[
    desarrollo_2_2["slug"].isin(slugs_recuperados)
].copy()
ejemplos_recuperados["fragmento_normalizado"] = ejemplos_recuperados["fragmento"].map(
    normalizar
)
ejemplos_recuperados["senales_originales"] = ejemplos_recuperados.apply(
    lambda fila: senales_fila(fila, "original"), axis=1
)
ejemplos_recuperados["senales_normalizadas"] = ejemplos_recuperados.apply(
    lambda fila: senales_fila(fila, "normalizada"), axis=1
)
print("Enlaces recuperados en desarrollo; no forman parte de las métricas:")
display(
    ejemplos_recuperados[
        [
            "slug",
            "title",
            "fragmento",
            "fragmento_normalizado",
            "senales_originales",
            "senales_normalizadas",
        ]
    ]
)

caso_homoglifos = desarrollo_2_2.loc[
    desarrollo_2_2["slug"].eq("mas-alla-13")
].copy()
caso_homoglifos["fragmento_normalizado"] = caso_homoglifos["fragmento"].map(
    normalizar
)
print("Homógrafos normalizados en un dominio desnudo, que la regex aprobada no marca:")
display(
    caso_homoglifos[
        [
            "slug",
            "title",
            "fragmento",
            "fragmento_normalizado",
            "marca_combinada_original",
            "marca_combinada_normalizada",
        ]
    ]
)

slugs_limpios_con_invisibles = ["hispania", "martin-lutero-ii"]
ejemplos_limpios_invisibles = desarrollo_2_2.loc[
    desarrollo_2_2["slug"].isin(slugs_limpios_con_invisibles)
].copy()
ejemplos_limpios_invisibles["caracteres_invisibles"] = ejemplos_limpios_invisibles[
    "synopsis"
].map(
    lambda texto: ", ".join(
        sorted({
            f"{unicodedata.name(caracter, 'SIN NOMBRE')} (U+{ord(caracter):04X})"
            for caracter in texto
            if unicodedata.category(caracter) == "Cf"
        })
    )
)
print("Sinopsis limpias de desarrollo con caracteres invisibles; no son contaminación por sí solos:")
display(
    ejemplos_limpios_invisibles[
        [
            "slug",
            "title",
            "caracteres_invisibles",
            "marca_combinada_original",
            "marca_combinada_normalizada",
        ]
    ]
)

Enlaces recuperados en desarrollo; no forman parte de las métricas:


,slug,title,fragmento,fragmento_normalizado,senales_originales,senales_normalizadas
280,espana-terra-incognita,"España, terra incognita",Link en nuestro sitio: h t t p s : / / lectorp...,link en nuestro sitio:https://lectorplus.invalid,,URL
283,los-crimenes-del-invisible,Los crímenes del invisible,Buscalo en w w w . libroplus . example,buscalo en www.libroplus.example,,URL


Homógrafos normalizados en un dominio desnudo, que la regex aprobada no marca:


,slug,title,fragmento,fragmento_normalizado,marca_combinada_original,marca_combinada_normalizada
263,mas-alla-13,Más Allá 13,Entrá a lеctorflаsh.test y leé sin límites.,entra a lectorflash.test y lee sin limites.,False,False


Sinopsis limpias de desarrollo con caracteres invisibles; no son contaminación por sí solos:


,slug,title,caracteres_invisibles,marca_combinada_original,marca_combinada_normalizada
428,hispania,Hispania,SOFT HYPHEN (U+00AD),False,False
132,martin-lutero-ii,Martín Lutero II,WORD JOINER (U+2060),False,False


La normalización elimina caracteres de formato (`Cf`) únicamente en la copia de trabajo. «Hispania» y «Martín Lutero II» muestran que puede haber caracteres invisibles en sinopsis limpias: por eso una marca requiere que alguna regex encuentre además una señal.

En «Más Allá 13» se convierten dos letras cirílicas visualmente similares a `e` y `a`, pero el texto contiene un dominio desnudo, sin `http(s)://` ni `www.`. Las regex del 2.1 siguen sin detectarlo; ampliarlas queda fuera de este apartado.

### 2.3 Detectar el idioma de cada oración

Primero se clasifica cada sinopsis completa. Después se divide el texto original en oraciones y se usa `detect_langs` para estimar el idioma principal de cada una. La partición de desarrollo sirve para elegir el mínimo de palabras y la probabilidad; la prueba se mide una sola vez con esos valores fijos.

Las oraciones cortas, los nombres propios y las citas pueden dar detecciones erróneas. Se comparan mínimos de 5, 7, 10, 12 y 15 palabras, y probabilidades de 0,90 y 0,99. El mínimo cuenta secuencias de letras y excluye números. La semilla fija permite repetir las probabilidades. Se muestran completas las oraciones para distinguir el idioma predicho del texto real. Las sinopsis originales no se modifican.

In [13]:
from langdetect import DetectorFactory, LangDetectException, detect_langs

DetectorFactory.seed = 42


def separar_oraciones(texto):
    partes = re.split(r"(?<=[.!?])\s+|\n", str(texto or ""))
    return [parte.strip() for parte in partes if parte.strip()]


def detectar_idioma(texto):
    try:
        idiomas = detect_langs(str(texto or ""))
    except LangDetectException:
        return "sin detección", 0.0
    if not idiomas:
        return "sin detección", 0.0
    principal = max(idiomas, key=lambda idioma: idioma.prob)
    return principal.lang, principal.prob


def analizar_oraciones(texto):
    resultados = []
    for oracion in separar_oraciones(texto):
        cantidad = len(re.findall(r"\b[^\W\d_]+\b", oracion, flags=re.UNICODE))
        if cantidad < 5:
            continue
        idioma, probabilidad = detectar_idioma(oracion)
        resultados.append({
            "oracion": oracion,
            "palabras": cantidad,
            "idioma": idioma,
            "probabilidad": probabilidad,
        })
    return resultados


def marcar_idioma_oracion(oraciones_analizadas, minimo_palabras, umbral):
    return any(
        oracion["palabras"] >= minimo_palabras
        and oracion["idioma"] not in ("es", "sin detección")
        and oracion["probabilidad"] >= umbral
        for oracion in oraciones_analizadas
    )

In [14]:
desarrollo_2_3 = desarrollo.copy()
idiomas_completos = desarrollo_2_3["synopsis"].map(detectar_idioma)
desarrollo_2_3["idioma_sinopsis"] = idiomas_completos.map(lambda resultado: resultado[0])
desarrollo_2_3["probabilidad_sinopsis"] = idiomas_completos.map(lambda resultado: resultado[1])
desarrollo_2_3["oraciones_analizadas"] = desarrollo_2_3["synopsis"].map(analizar_oraciones)

tabla_idioma_completo = pd.crosstab(
    desarrollo_2_3["contaminado"],
    desarrollo_2_3["idioma_sinopsis"],
)
print("Idioma principal de la sinopsis completa en desarrollo:")
display(tabla_idioma_completo)

minimos_comparados = [5, 7, 10, 12, 15]
umbrales_comparados = [0.90, 0.99]
y_desarrollo_idioma = desarrollo_2_3["contaminado"].astype(int)
resultados_umbrales = []
for minimo in minimos_comparados:
    for umbral in umbrales_comparados:
        marcas = desarrollo_2_3["oraciones_analizadas"].map(
            lambda oraciones: marcar_idioma_oracion(oraciones, minimo, umbral)
        )
        resultados_umbrales.append({
            "minimo_palabras": minimo,
            "probabilidad_minima": umbral,
            "precision": round(precision_score(y_desarrollo_idioma, marcas, zero_division=0), 2),
            "recall": round(recall_score(y_desarrollo_idioma, marcas, zero_division=0), 2),
            "contaminados_marcados": int((marcas & y_desarrollo_idioma.eq(1)).sum()),
            "limpios_marcados": int((marcas & y_desarrollo_idioma.eq(0)).sum()),
        })

tabla_umbrales_idioma = pd.DataFrame(resultados_umbrales)
display(tabla_umbrales_idioma)

MINIMO_PALABRAS_IDIOMA = 10
UMBRAL_IDIOMA = 0.90
desarrollo_2_3["marca_idioma_oracion"] = desarrollo_2_3["oraciones_analizadas"].map(
    lambda oraciones: marcar_idioma_oracion(
        oraciones,
        MINIMO_PALABRAS_IDIOMA,
        UMBRAL_IDIOMA,
    )
)
print(
    f"Parámetros elegidos solo con desarrollo: mínimo {MINIMO_PALABRAS_IDIOMA} palabras "
    f"y probabilidad mínima {UMBRAL_IDIOMA:.2f}."
)

Idioma principal de la sinopsis completa en desarrollo:


idioma_sinopsis,es
contaminado,
0,280
1,70


,minimo_palabras,probabilidad_minima,precision,recall,contaminados_marcados,limpios_marcados
0,5,0.90,0.56,0.21,15,12
1,5,0.99,0.56,0.21,15,12
2,7,0.90,0.80,0.17,12,3
3,7,0.99,0.80,0.17,12,3
4,10,0.90,0.82,0.13,9,2
5,10,0.99,0.82,0.13,9,2
6,12,0.90,0.78,0.10,7,2
7,12,0.99,0.78,0.10,7,2
8,15,0.90,0.00,0.00,0,1
9,15,0.99,0.00,0.00,0,1


Parámetros elegidos solo con desarrollo: mínimo 10 palabras y probabilidad mínima 0.90.


En desarrollo, 10 palabras y 0,90 marcan 9 de 70 libros contaminados y 2 de 280 limpios (precisión 0,82 y recall 0,13). Con 7 palabras se detectan 12 contaminados y 3 limpios; con 15 palabras no se detectan contaminados. Subir la probabilidad a 0,99 no cambia los resultados: `langdetect` asigna probabilidades cercanas a 1,0 incluso a algunas oraciones españolas mal clasificadas. Se eligen 10 palabras y 0,90 para reducir falsos positivos conservando detecciones de mensajes en inglés y portugués.

Las 350 sinopsis completas quedan clasificadas como español, incluidas las 70 contaminadas. Las etiquetas indican contaminación, no idioma: «Activá tu prueba gratuita desde tu celular» es una frase española que se clasificó como rumano con el mínimo anterior de 7 palabras. Con el mínimo actual de 10 queda excluida. Por eso, la métrica evalúa coincidencia con la etiqueta de contaminación; no mide directamente la exactitud del idioma detectado.

In [15]:
def filas_oraciones_marcadas(datos):
    filas = []
    for _, fila in datos.iterrows():
        for oracion in fila["oraciones_analizadas"]:
            if (
                oracion["palabras"] >= MINIMO_PALABRAS_IDIOMA
                and oracion["idioma"] not in ("es", "sin detección")
                and oracion["probabilidad"] >= UMBRAL_IDIOMA
            ):
                filas.append({
                    "slug": fila["slug"],
                    "title": fila["title"],
                    "fragmento_insertado": fila.get("fragmento", ""),
                    "contaminado": int(fila["contaminado"]),
                    "idioma": oracion["idioma"],
                    "probabilidad": round(oracion["probabilidad"], 2),
                    "palabras": oracion["palabras"],
                    "oracion": oracion["oracion"],
                })
    return pd.DataFrame(filas)

oraciones_marcadas_desarrollo = filas_oraciones_marcadas(desarrollo_2_3)
print("Candidatas en libros contaminados de desarrollo; idioma y probabilidad son predicciones:")
columnas_oraciones_idioma = [
    "slug", "title", "idioma", "probabilidad", "palabras",
    "fragmento_insertado", "oracion",
]
with pd.option_context("display.max_colwidth", None):
    display(
        oraciones_marcadas_desarrollo.loc[
            oraciones_marcadas_desarrollo["contaminado"].eq(1),
            columnas_oraciones_idioma,
        ]
    )

falsos_positivos_idioma_desarrollo = oraciones_marcadas_desarrollo.loc[
    oraciones_marcadas_desarrollo["contaminado"].eq(0)
]
print(
    "Falsos positivos en libros limpios de desarrollo: "
    f"{len(falsos_positivos_idioma_desarrollo)} oraciones. "
    "Son textos limpios clasificados erróneamente como otro idioma."
)
with pd.option_context("display.max_colwidth", None):
    display(
        falsos_positivos_idioma_desarrollo[
            ["slug", "title", "idioma", "probabilidad", "palabras", "oracion"]
        ]
    )

Candidatas en libros contaminados de desarrollo; idioma y probabilidad son predicciones:


,slug,title,idioma,probabilidad,palabras,fragmento_insertado,oracion
1,el-agrio,El Agrio,en,1.0,13,"Download this ebook for FREE, no sign-up required, at biblioya.test today.","Download this ebook for FREE, no sign-up required, at biblioya.test today."
2,tradiciones-peruanas,Tradiciones peruanas,pt,1.0,11,"Baixe este livro grátis agora mesmo, sem cadastro, em librohub.test.","Baixe este livro grátis agora mesmo, sem cadastro, em librohub.test."
4,el-cerebro-bilingue,El cerebro bilingüe,en,1.0,13,"Download this ebook for FREE, no sign-up required, at relatoflash.test today.","Download this ebook for FREE, no sign-up required, at relatoflash.test today."
5,ciclos-del-tiempo,Ciclos del tiempo,en,1.0,13,"Download this ebook for FREE, no sign-up required, at lectorya.test today.","Download this ebook for FREE, no sign-up required, at lectorya.test today."
6,los-asesinos-fredric-brown,Los asesinos,en,1.0,13,"Download this ebook for FREE, no sign-up required, at capitulototal.invalid today.","Download this ebook for FREE, no sign-up required, at capitulototal.invalid today."
7,la-venganza-del-caballero-halcon,La venganza del caballero Halcón,en,1.0,14,Conseguí este título y más de 50.000 libros en la app LibroLibre: el primer mes es sin cargo.,SE ACUSA AL CABALLERO HALCÓN DE HABER INTENTADO ASESINAR AL REY ENRIQUE: ¿SERÁ CIERTO?
8,poemas-dispersos,Poemas dispersos,pt,1.0,14,Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!,Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!
9,opiniones-de-un-payaso,Opiniones de un payaso,en,1.0,12,Get unlimited access to thousands of bestsellers with our limited time offer!,Get unlimited access to thousands of bestsellers with our limited time offer!
10,la-batalla-del-marne,La batalla del Marne,it,1.0,11,Conseguí este título y más de 50.000 libros en la app LibroLibre: el primer mes es sin cargo.,"De esta manera, la batalla del Marne pone fin a la"


Falsos positivos en libros limpios de desarrollo: 2 oraciones. Son textos limpios clasificados erróneamente como otro idioma.


,slug,title,idioma,probabilidad,palabras,oracion
0,mossflower,Mossflower,ca,1.0,12,Es ésta la segunda entrega del ciclo que se inicia con Redwall.
3,la-amenaza-del-dueno-del-tiempo,La amenaza del dueño del tiempo,de,1.0,16,LA HIJA DEL REY ENRIQUE SE HA PERDIDO EN LAS ENTRAÑAS DE LAS CAVERNAS DEL TIEMPO.


### Medición final del apartado 2.3

Una vez elegidos los valores con desarrollo, se aplican sin cambios a los 150 libros de prueba. La marca de sinopsis completa compara el idioma principal con español; la marca por oración requiere una oración de al menos 10 palabras cuyo idioma principal no sea español y cuya probabilidad sea de al menos 0,90.

In [16]:
prueba_2_3 = prueba.copy()
idiomas_prueba_completos = prueba_2_3["synopsis"].map(detectar_idioma)
prueba_2_3["idioma_sinopsis"] = idiomas_prueba_completos.map(lambda resultado: resultado[0])
prueba_2_3["marca_idioma_sinopsis"] = ~prueba_2_3["idioma_sinopsis"].isin(("es", "sin detección"))
prueba_2_3["oraciones_analizadas"] = prueba_2_3["synopsis"].map(analizar_oraciones)
prueba_2_3["marca_idioma_oracion"] = prueba_2_3["oraciones_analizadas"].map(
    lambda oraciones: marcar_idioma_oracion(
        oraciones,
        MINIMO_PALABRAS_IDIOMA,
        UMBRAL_IDIOMA,
    )
)

y_prueba_idioma = prueba_2_3["contaminado"].astype(int)
resultados_idioma = []
for nombre, columna in [
    ("Sinopsis completa", "marca_idioma_sinopsis"),
    ("Oraciones", "marca_idioma_oracion"),
]:
    predicciones = prueba_2_3[columna].astype(int)
    resultados_idioma.append({
        "detector": nombre,
        "precision": round(precision_score(y_prueba_idioma, predicciones, zero_division=0), 2),
        "recall": round(recall_score(y_prueba_idioma, predicciones, zero_division=0), 2),
        "libros_marcados": int(predicciones.sum()),
    })

tabla_metricas_idioma = pd.DataFrame(resultados_idioma)
display(tabla_metricas_idioma)

oraciones_marcadas_prueba = filas_oraciones_marcadas(prueba_2_3)
oraciones_contaminadas_prueba = oraciones_marcadas_prueba.loc[
    oraciones_marcadas_prueba["contaminado"].eq(1)
]
print("Candidatas en libros contaminados de prueba; idioma y probabilidad son predicciones:")
with pd.option_context("display.max_colwidth", None):
    display(
        oraciones_contaminadas_prueba[
            ["slug", "title", "idioma", "probabilidad", "palabras", "fragmento_insertado", "oracion"]
        ]
    )

falsos_positivos_idioma_prueba = oraciones_marcadas_prueba.loc[
    oraciones_marcadas_prueba["contaminado"].eq(0)
]
print(
    "\n\nFalsos positivos de libros limpios en prueba: "
    f"{len(falsos_positivos_idioma_prueba)} oraciones. "
    "\nEl idioma es una predicción; el texto completo permite revisar el error."
)
if falsos_positivos_idioma_prueba.empty:
    print("No hubo falsos positivos por oración en prueba.")
else:
    with pd.option_context("display.max_colwidth", None):
        display(
            falsos_positivos_idioma_prueba[
                ["slug", "title", "idioma", "probabilidad", "palabras", "oracion"]
            ]
        )

,detector,precision,recall,libros_marcados
0,Sinopsis completa,0.00,0.00,0
1,Oraciones,0.89,0.27,9


Candidatas en libros contaminados de prueba; idioma y probabilidad son predicciones:


,slug,title,idioma,probabilidad,palabras,fragmento_insertado,oracion
0,trejo,Trejo,en,1.0,13,"Download this ebook for FREE, no sign-up required, at paginazona.test today.","Download this ebook for FREE, no sign-up required, at paginazona.test today."
2,el-imperio-final-ed-ilustrada,El Imperio Final (Ed. ilustrada),en,1.0,13,"Download this ebook for FREE, no sign-up required, at libroclub.invalid today.","Download this ebook for FREE, no sign-up required, at libroclub.invalid today."
3,el-tren-de-jaen,El tren de Jaén,pt,1.0,14,Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!,Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!
4,la-nina-que-amaba-las-cerillas,La niña que amaba las cerillas,pt,1.0,11,"Baixe este livro grátis agora mesmo, sem cadastro, em epubzona.invalid.","Baixe este livro grátis agora mesmo, sem cadastro, em epubzona.invalid."
5,historia-de-la-violacion-siglos-xvi-xx,Historia de la violacion: Siglos XVI-XX,en,1.0,12,Get unlimited access to thousands of bestsellers with our limited time offer!,Get unlimited access to thousands of bestsellers with our limited time offer!
6,umbra-vitae,Umbra vitae,pt,1.0,14,Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!,Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!
7,un-corazon-habitado-por-mil-voces,Un corazón habitado por mil voces,pt,1.0,14,Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!,Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!
8,la-revolucion-de-la-glucosa,La revolución de la glucosa,pt,1.0,14,Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!,Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!




Falsos positivos de libros limpios en prueba: 1 oraciones. 
El idioma es una predicción; el texto completo permite revisar el error.


,slug,title,idioma,probabilidad,palabras,oracion
1,el-rey-de-amarillo,El rey de amarillo,en,1.0,30,"Sus obras más importantes en el campo de la fantasía son las siguientes: The King in Yellow (1895), The Maker of Moons (1896), The Mystery of Choice (1897), In Search of Unknown (1904) Police!!!"


### 2.4 Reconocer entidades

Se aplica spaCy con `es_core_news_lg` a las oraciones de los fragmentos insertados en libros contaminados de desarrollo y a la misma cantidad de oraciones de sinopsis limpias. La selección limpia es reproducible y la prueba no se utiliza. Se comparan las entidades y sus etiquetas; una entidad exclusiva de esta muestra no necesariamente es exclusiva de toda contaminación.


In [17]:
import spacy

nlp_ner = spacy.load("es_core_news_lg")

oraciones_fragmentos_ner = []
for _, fila in desarrollo.loc[desarrollo["contaminado"].eq(1)].iterrows():
    for oracion in separar_oraciones(fila["fragmento"]):
        if len(oracion) >= 15:
            oraciones_fragmentos_ner.append({
                "grupo": "Fragmento insertado",
                "slug": fila["slug"],
                "title": fila["title"],
                "oracion": oracion,
            })

oraciones_limpias_ner = []
for _, fila in desarrollo.loc[desarrollo["contaminado"].eq(0)].iterrows():
    for oracion in separar_oraciones(fila["synopsis"]):
        if len(oracion) >= 15:
            oraciones_limpias_ner.append({
                "grupo": "Sinopsis limpia",
                "slug": fila["slug"],
                "title": fila["title"],
                "oracion": oracion,
            })

oraciones_limpias_ner = (
    pd.DataFrame(oraciones_limpias_ner)
    .drop_duplicates(subset="oracion")
    .sample(n=len(oraciones_fragmentos_ner), random_state=42)
    .reset_index(drop=True)
)
oraciones_comparadas_ner = pd.concat(
    [pd.DataFrame(oraciones_fragmentos_ner), oraciones_limpias_ner],
    ignore_index=True,
)
print(
    f"Oraciones comparadas: {len(oraciones_fragmentos_ner)} de fragmentos insertados "
    f"y {len(oraciones_limpias_ner)} limpias."
)

filas_entidades_ner = []
for id_oracion, fila in enumerate(oraciones_comparadas_ner.to_dict("records")):
    doc_ner = nlp_ner(fila["oracion"])
    for entidad in doc_ner.ents:
        filas_entidades_ner.append({
            "id_oracion": id_oracion,
            "grupo": fila["grupo"],
            "slug": fila["slug"],
            "title": fila["title"],
            "oracion": fila["oracion"],
            "entidad": entidad.text,
            "etiqueta": entidad.label_,
            "explicacion": spacy.explain(entidad.label_) or entidad.label_,
        })

entidades_ner = pd.DataFrame(filas_entidades_ner)
resumen_entidades_ner = []
for grupo in ["Fragmento insertado", "Sinopsis limpia"]:
    oraciones_grupo = oraciones_comparadas_ner.loc[
        oraciones_comparadas_ner["grupo"].eq(grupo)
    ]
    entidades_grupo = entidades_ner.loc[entidades_ner["grupo"].eq(grupo)]
    oraciones_con_entidades = entidades_grupo["id_oracion"].nunique()
    resumen_entidades_ner.append({
        "grupo": grupo,
        "oraciones": len(oraciones_grupo),
        "oraciones_con_entidades": oraciones_con_entidades,
        "porcentaje_con_entidades": round(100 * oraciones_con_entidades / len(oraciones_grupo), 1),
        "menciones": len(entidades_grupo),
        "entidades_distintas": entidades_grupo[["entidad", "etiqueta"]].drop_duplicates().shape[0],
    })
display(pd.DataFrame(resumen_entidades_ner))

if entidades_ner.empty:
    print("El modelo no reconoció entidades en las oraciones comparadas.")
else:
    grupos_ner = ["Fragmento insertado", "Sinopsis limpia"]
    entidades_ner["entidad_normalizada"] = entidades_ner["entidad"].str.casefold()
    conteos_entidades_ner = entidades_ner.groupby(
        ["entidad_normalizada", "etiqueta", "grupo"]
    ).size().unstack("grupo", fill_value=0)
    for grupo in grupos_ner:
        if grupo not in conteos_entidades_ner.columns:
            conteos_entidades_ner[grupo] = 0
    conteos_entidades_ner = conteos_entidades_ner[grupos_ner]
    formas_entidades_ner = entidades_ner.groupby(
        ["entidad_normalizada", "etiqueta"]
    )["entidad"].first()
    tabla_entidades_ner = conteos_entidades_ner.copy()
    tabla_entidades_ner.insert(
        0,
        "entidad",
        [formas_entidades_ner.loc[clave] for clave in tabla_entidades_ner.index],
    )
    tabla_entidades_ner = tabla_entidades_ner.reset_index()
    display(
        tabla_entidades_ner.loc[
            tabla_entidades_ner["Fragmento insertado"].gt(0)
            & tabla_entidades_ner["Sinopsis limpia"].eq(0),
            ["entidad", "etiqueta", "Fragmento insertado", "Sinopsis limpia"],
        ].sort_values(["Fragmento insertado", "entidad"], ascending=[False, True])
    )
    print("Entidades detectadas también en ambos grupos:")
    display(
        tabla_entidades_ner.loc[
            tabla_entidades_ner["Fragmento insertado"].gt(0)
            & tabla_entidades_ner["Sinopsis limpia"].gt(0),
            ["entidad", "etiqueta", "Fragmento insertado", "Sinopsis limpia"],
        ].sort_values(["Fragmento insertado", "Sinopsis limpia"], ascending=False)
    )
    print("Ejemplos de entidades reconocidas en su oración:")
    with pd.option_context("display.max_colwidth", 100):
        display(
            entidades_ner[
                ["grupo", "slug", "entidad", "etiqueta", "oracion"]
            ].groupby("grupo", group_keys=False).head(5)
        )


Oraciones comparadas: 88 de fragmentos insertados y 88 limpias.


,grupo,oraciones,oraciones_con_entidades,porcentaje_con_entidades,menciones,entidades_distintas
0,Fragmento insertado,88,51,58.0,70,39
1,Sinopsis limpia,88,36,40.9,55,54


grupo,entidad,etiqueta,Fragmento insertado,Sinopsis limpia
11,compartila,PER,7,0
63,PDF,MISC,6,0
28,Ganá,PER,5,0
14,Descargalo,MISC,4,0
15,Download this ebook for FREE,MISC,4,0
19,EPUB sin registro,MISC,4,0
73,sign-up required,MISC,4,0
18,EPUB,MISC,2,0
20,EpubMax,MISC,2,0
53,LibroLibre,MISC,2,0


Entidades detectadas también en ambos grupos:


grupo,entidad,etiqueta,Fragmento insertado,Sinopsis limpia


Ejemplos de entidades reconocidas en su oración:


,grupo,slug,entidad,etiqueta,oracion
0,Fragmento insertado,el-agrio,Download this ebook for FREE,MISC,"Download this ebook for FREE, no sign-up required, at biblioya.test today."
1,Fragmento insertado,el-agrio,sign-up required,MISC,"Download this ebook for FREE, no sign-up required, at biblioya.test today."
2,Fragmento insertado,el-camino-de-la-herencia,Visitá pagin⁠azona.invalid,LOC,Visitá pagin⁠azona.invalid para la versión completa.
3,Fragmento insertado,tradiciones-peruanas,Baixe,LOC,"Baixe este livro grátis agora mesmo, sem cadastro, em librohub.test."
4,Fragmento insertado,tradiciones-peruanas,grátis agora mesmo,LOC,"Baixe este livro grátis agora mesmo, sem cadastro, em librohub.test."
70,Sinopsis limpia,el-ritmo-de-la-noche,Lee Parker,PER,"Pero entonces conoce a Lee Parker, el hijo de una de las familias más respetadas de la ciudad."
71,Sinopsis limpia,el-templo-del-alba,Japón,LOC,"A su regreso al Japón, Honda se vuelve espectador."
72,Sinopsis limpia,el-templo-del-alba,Honda,ORG,"A su regreso al Japón, Honda se vuelve espectador."
73,Sinopsis limpia,escritos-sobre-literatura,Baudelaire,PER,"Baudelaire, en estos escritos, lleva a cabo un análisis admirable de una penetración tan rara en..."
74,Sinopsis limpia,la-busqueda-de-los-elementos,Tales de Mileto,PER,"Desde Tales de Mileto hasta Seaborg de California, de la alquimia al ciclotrón, desde la búsqued..."


**Interpretación del apartado.** En las 88 oraciones de cada grupo, spaCy reconoció al menos una entidad en 51 oraciones de fragmentos insertados (58,0 %) y en 36 oraciones limpias (40,9 %). Encontró 39 entidades distintas en los fragmentos y 54 en las sinopsis limpias; ninguna pareja exacta de entidad y etiqueta apareció en ambos grupos de esta muestra. Entre las detectadas solo en fragmentos aparecen nombres promocionales como `EpubMax`, `LibroLibre` y `TomoClub`, señales como `PDF`, y textos de descarga o registro. El modelo también etiqueta algunos imperativos como `PER` o `LOC` y detecta URLs o correos como otras categorías, lo que muestra sus límites.

**¿Sirven para detectar o solo para explicar lo que encontraron otros detectores?** Estos resultados sirven principalmente para explicar y describir fragmentos: destacan nombres de sitios, formatos y mensajes promocionales. La presencia de entidades por sí sola no separa bien los grupos: 36 de 88 oraciones limpias también contienen entidades, y la muestra limpia puede no incluir entidades legítimas que sí aparecen en otros libros. Unidad 3 advierte que el modelo en español puede omitir entidades o marcar falsas. Por eso, en este apartado NER aporta contexto para revisar señales detectadas por otros métodos; no se usa como regla automática ni se afirma que mejore la detección.

### 2.5 Clasificar oraciones

Las oraciones de `fragmento` de los libros contaminados de desarrollo son ejemplos positivos; las oraciones de sinopsis limpias son negativos. Separamos los libros de desarrollo por `slug` en ajuste y calibración, así ninguna oración del mismo libro aparece en ambos conjuntos. Entrenamos regresión logística con TF-IDF y con embeddings multilingües. La probabilidad de un libro es la máxima probabilidad de sus oraciones; el umbral se elige en calibración con `precision_recall_curve`, intentando alcanzar precisión 0,90. La prueba exterior queda reservada para la medición final.


In [18]:
import numpy as np
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import train_test_split

entrenamiento_2_5, calibracion_2_5 = train_test_split(
    desarrollo,
    test_size=0.3,
    random_state=42,
    stratify=desarrollo["contaminado"],
)

resumen_desarrollo_2_5 = pd.DataFrame([
    {
        "particion": nombre,
        "libros": len(datos),
        "limpios": int(datos["contaminado"].eq(0).sum()),
        "contaminados": int(datos["contaminado"].eq(1).sum()),
    }
    for nombre, datos in [
        ("ajuste", entrenamiento_2_5),
        ("calibración", calibracion_2_5),
    ]
])
display(resumen_desarrollo_2_5)


def armar_oraciones_entrenamiento_2_5(datos):
    filas = []
    for _, fila in datos.iterrows():
        etiqueta = int(fila["contaminado"])
        texto = fila["fragmento"] if etiqueta == 1 else fila["synopsis"]
        for oracion in separar_oraciones(texto):
            if len(oracion) >= 15:
                filas.append({
                    "slug": fila["slug"],
                    "oracion": oracion,
                    "contaminado": etiqueta,
                })
    return pd.DataFrame(filas)


def armar_oraciones_sinopsis_2_5(datos):
    filas = []
    for _, fila in datos.iterrows():
        for oracion in separar_oraciones(fila["synopsis"]):
            if len(oracion) >= 15:
                filas.append({
                    "slug": fila["slug"],
                    "title": fila["title"],
                    "oracion": oracion,
                    "contaminado": int(fila["contaminado"]),
                })
    return pd.DataFrame(filas)


oraciones_entrenamiento_2_5 = armar_oraciones_entrenamiento_2_5(entrenamiento_2_5)
print("Oraciones usadas para ajustar los modelos:")
display(
    oraciones_entrenamiento_2_5["contaminado"]
    .value_counts()
    .rename(index={0: "Sinopsis limpia (negativa)", 1: "Fragmento (positiva)"})
    .rename("oraciones")
    .to_frame()
)

vectorizador_tfidf_2_5 = TfidfVectorizer()
x_tfidf_2_5 = vectorizador_tfidf_2_5.fit_transform(
    oraciones_entrenamiento_2_5["oracion"]
)
clasificador_tfidf_2_5 = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
)
clasificador_tfidf_2_5.fit(
    x_tfidf_2_5,
    oraciones_entrenamiento_2_5["contaminado"],
)

NOMBRE_MODELO_EMBEDDINGS_2_5 = "distiluse-base-multilingual-cased-v1"
modelo_embeddings_2_5 = SentenceTransformer(NOMBRE_MODELO_EMBEDDINGS_2_5)
textos_entrenamiento_2_5 = oraciones_entrenamiento_2_5["oracion"].tolist()
embeddings_entrenamiento_2_5 = modelo_embeddings_2_5.encode(
    textos_entrenamiento_2_5,
    convert_to_numpy=True,
    show_progress_bar=True,
)
clasificador_embeddings_2_5 = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
)
clasificador_embeddings_2_5.fit(
    embeddings_entrenamiento_2_5,
    oraciones_entrenamiento_2_5["contaminado"],
)


def vectorizar_tfidf_2_5(textos):
    return vectorizador_tfidf_2_5.transform(textos)


def vectorizar_embeddings_2_5(textos):
    return modelo_embeddings_2_5.encode(
        textos,
        convert_to_numpy=True,
        show_progress_bar=False,
    )


def puntajes_por_libro_2_5(datos, clasificador, vectorizar):
    oraciones = armar_oraciones_sinopsis_2_5(datos)
    vectores = vectorizar(oraciones["oracion"].tolist())
    columna_positiva = list(clasificador.classes_).index(1)
    oraciones["probabilidad_maxima"] = clasificador.predict_proba(vectores)[
        :, columna_positiva
    ]
    indices_maximos = oraciones.groupby("slug")["probabilidad_maxima"].idxmax()
    return oraciones.loc[
        indices_maximos,
        ["slug", "title", "contaminado", "oracion", "probabilidad_maxima"],
    ].rename(columns={"oracion": "oracion_mas_sospechosa"}).reset_index(drop=True)


puntajes_calibracion_tfidf_2_5 = puntajes_por_libro_2_5(
    calibracion_2_5,
    clasificador_tfidf_2_5,
    vectorizar_tfidf_2_5,
)
puntajes_calibracion_embeddings_2_5 = puntajes_por_libro_2_5(
    calibracion_2_5,
    clasificador_embeddings_2_5,
    vectorizar_embeddings_2_5,
)

precision_tfidf_2_5, recall_tfidf_2_5, umbrales_tfidf_2_5 = precision_recall_curve(
    puntajes_calibracion_tfidf_2_5["contaminado"],
    puntajes_calibracion_tfidf_2_5["probabilidad_maxima"],
)
precision_embeddings_2_5, recall_embeddings_2_5, umbrales_embeddings_2_5 = precision_recall_curve(
    puntajes_calibracion_embeddings_2_5["contaminado"],
    puntajes_calibracion_embeddings_2_5["probabilidad_maxima"],
)


def maxima_precision_con_detecciones_2_5(precision, recall):
    precisiones_con_positivos = precision[:-1][recall[:-1] > 0]
    return float(precisiones_con_positivos.max())


maxima_tfidf_2_5 = maxima_precision_con_detecciones_2_5(
    precision_tfidf_2_5,
    recall_tfidf_2_5,
)
maxima_embeddings_2_5 = maxima_precision_con_detecciones_2_5(
    precision_embeddings_2_5,
    recall_embeddings_2_5,
)
OBJETIVO_PRECISION_2_5 = min(0.90, maxima_tfidf_2_5, maxima_embeddings_2_5)


def elegir_umbral_2_5(precision, recall, umbrales, objetivo):
    candidatos = np.flatnonzero(
        (precision[:-1] >= objetivo - 1e-12)
        & (recall[:-1] > 0)
    )
    mejor = candidatos[np.argmax(recall[:-1][candidatos])]
    return {
        "umbral": float(umbrales[mejor]),
        "precision_calibracion": float(precision[mejor]),
        "recall_calibracion": float(recall[mejor]),
    }


seleccion_umbral_tfidf_2_5 = elegir_umbral_2_5(
    precision_tfidf_2_5,
    recall_tfidf_2_5,
    umbrales_tfidf_2_5,
    OBJETIVO_PRECISION_2_5,
)
seleccion_umbral_embeddings_2_5 = elegir_umbral_2_5(
    precision_embeddings_2_5,
    recall_embeddings_2_5,
    umbrales_embeddings_2_5,
    OBJETIVO_PRECISION_2_5,
)
umbrales_modelos_2_5 = {
    "TF-IDF": seleccion_umbral_tfidf_2_5["umbral"],
    "Embeddings": seleccion_umbral_embeddings_2_5["umbral"],
}

print(
    f"Objetivo común de precisión en desarrollo: {OBJETIVO_PRECISION_2_5:.2f}. "
    "Se intenta 0,90; si alguno no llega, se usa el máximo común alcanzable."
)
display(pd.DataFrame([
    {
        "representacion": nombre,
        **seleccion,
        "objetivo_comun": OBJETIVO_PRECISION_2_5,
    }
    for nombre, seleccion in [
        ("TF-IDF", seleccion_umbral_tfidf_2_5),
        ("Embeddings", seleccion_umbral_embeddings_2_5),
    ]
]))


C:\Users\Tomas\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,particion,libros,limpios,contaminados
0,ajuste,245,196,49
1,calibración,105,84,21


Oraciones usadas para ajustar los modelos:


,oraciones
contaminado,
Sinopsis limpia (negativa),1387
Fragmento (positiva),61


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Batches:  98%|█████████▊| 45/46 [00:32<00:00,  4.27it/s]

Batches: 100%|██████████| 46/46 [00:32<00:00,  4.99it/s]

Batches: 100%|██████████| 46/46 [00:32<00:00,  1.40it/s]

Objetivo común de precisión en desarrollo: 0.90. Se intenta 0,90; si alguno no llega, se usa el máximo común alcanzable.


,representacion,umbral,precision_calibracion,recall_calibracion,objetivo_comun
0,TF-IDF,0.641574,0.928571,0.619048,0.9
1,Embeddings,0.789196,0.933333,0.666667,0.9


In [19]:
from sklearn.metrics import precision_score, recall_score

puntajes_prueba_tfidf_2_5 = puntajes_por_libro_2_5(
    prueba,
    clasificador_tfidf_2_5,
    vectorizar_tfidf_2_5,
)
puntajes_prueba_embeddings_2_5 = puntajes_por_libro_2_5(
    prueba,
    clasificador_embeddings_2_5,
    vectorizar_embeddings_2_5,
)
comparacion_prueba_2_5 = puntajes_prueba_tfidf_2_5.merge(
    puntajes_prueba_embeddings_2_5,
    on=["slug", "title", "contaminado"],
    suffixes=("_tfidf", "_embeddings"),
).merge(
    prueba[["slug", "fragmento"]],
    on="slug",
    how="left",
)
fragmentos_de_ajuste_2_5 = set(
    entrenamiento_2_5.loc[
        entrenamiento_2_5["contaminado"].eq(1), "fragmento"
    ].astype(str).str.strip()
)
comparacion_prueba_2_5["fragmento_visto_en_ajuste"] = (
    comparacion_prueba_2_5["fragmento"].astype(str).str.strip().isin(fragmentos_de_ajuste_2_5)
)
comparacion_prueba_2_5["marca_tfidf"] = (
    comparacion_prueba_2_5["probabilidad_maxima_tfidf"]
    >= umbrales_modelos_2_5["TF-IDF"]
)
comparacion_prueba_2_5["marca_embeddings"] = (
    comparacion_prueba_2_5["probabilidad_maxima_embeddings"]
    >= umbrales_modelos_2_5["Embeddings"]
)
comparacion_prueba_2_5["coincidencia"] = np.select(
    [
        comparacion_prueba_2_5["marca_tfidf"] & comparacion_prueba_2_5["marca_embeddings"],
        comparacion_prueba_2_5["marca_tfidf"],
        comparacion_prueba_2_5["marca_embeddings"],
    ],
    ["Ambos", "Solo TF-IDF", "Solo embeddings"],
    default="Ninguno",
)

y_prueba_clasificacion_2_5 = comparacion_prueba_2_5["contaminado"].astype(int)
resultados_prueba_clasificacion_2_5 = []
for nombre, columna in [
    ("TF-IDF", "marca_tfidf"),
    ("Embeddings", "marca_embeddings"),
]:
    predicciones = comparacion_prueba_2_5[columna].astype(int)
    resultados_prueba_clasificacion_2_5.append({
        "representacion": nombre,
        "precision": round(
            precision_score(y_prueba_clasificacion_2_5, predicciones, zero_division=0),
            2,
        ),
        "recall": round(
            recall_score(y_prueba_clasificacion_2_5, predicciones, zero_division=0),
            2,
        ),
        "libros_marcados": int(predicciones.sum()),
        "objetivo_precision_desarrollo": OBJETIVO_PRECISION_2_5,
        "umbral": umbrales_modelos_2_5[nombre],
    })
print("Comparación en los mismos 150 libros de prueba, a nivel libro:")
display(pd.DataFrame(resultados_prueba_clasificacion_2_5))

print("Coincidencia de los detectores entre libros contaminados de prueba:")
display(
    comparacion_prueba_2_5.loc[
        comparacion_prueba_2_5["contaminado"].eq(1), "coincidencia"
    ].value_counts().rename("libros").to_frame()
)

positivos_prueba_clasificacion_2_5 = comparacion_prueba_2_5.loc[
    comparacion_prueba_2_5["contaminado"].eq(1)
].copy()
print("Fragmentos contaminados de prueba y resultado de cada clasificador:")
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(
        positivos_prueba_clasificacion_2_5[
            [
                "slug",
                "title",
                "fragmento",
                "fragmento_visto_en_ajuste",
                "coincidencia",
                "probabilidad_maxima_tfidf",
                "probabilidad_maxima_embeddings",
            ]
        ].sort_values(["coincidencia", "slug"])
    )

falsos_positivos_clasificacion_2_5 = comparacion_prueba_2_5.loc[
    comparacion_prueba_2_5["contaminado"].eq(0)
    & (
        comparacion_prueba_2_5["marca_tfidf"]
        | comparacion_prueba_2_5["marca_embeddings"]
    )
]
print("Falsos positivos de prueba y oración con mayor probabilidad:")
if falsos_positivos_clasificacion_2_5.empty:
    print("Ningún libro limpio fue marcado por los dos clasificadores.")
else:
    with pd.option_context("display.max_colwidth", None):
        display(
            falsos_positivos_clasificacion_2_5[
                [
                    "slug",
                    "title",
                    "marca_tfidf",
                    "oracion_mas_sospechosa_tfidf",
                    "marca_embeddings",
                    "oracion_mas_sospechosa_embeddings",
                ]
            ]
        )


Comparación en los mismos 150 libros de prueba, a nivel libro:


,representacion,precision,recall,libros_marcados,objetivo_precision_desarrollo,umbral
0,TF-IDF,1.0,0.57,17,0.9,0.641574
1,Embeddings,1.0,0.73,22,0.9,0.789196


Coincidencia de los detectores entre libros contaminados de prueba:


,libros
coincidencia,
Ambos,16
Ninguno,7
Solo embeddings,6
Solo TF-IDF,1


Fragmentos contaminados de prueba y resultado de cada clasificador:


,slug,title,fragmento,fragmento_visto_en_ajuste,coincidencia,probabilidad_maxima_tfidf,probabilidad_maxima_embeddings
7,bajo-la-isla,Bajo la isla,Una obra imprescindible que no puede faltar en tu biblioteca personal. Más títulos como este en www.librolibre.example,False,Ambos,0.762748,0.880492
8,bajo-la-isla-2,Bajo la isla,"Una historia apasionante que te atrapará desde la primera página hasta el final. Ganá dinero desde tu casa recomendando libros. Cupos limitados, consultá requisitos.",False,Ambos,0.943109,0.944812
16,come-as-you-are-la-historia-de-nirvana,Come as you are: la historia de Nirvana,"Si te gustó esta reseña, compartila con tus amigos y participá del sorteo de un lector electrónico.",True,Ambos,0.929258,0.949192
39,el-imperio-final-ed-ilustrada,El Imperio Final (Ed. ilustrada),"Download this ebook for FREE, no sign-up required, at libroclub.invalid today.",False,Ambos,0.838960,0.972611
47,el-tren-de-las-310-a-yuma-y-otros-relatos-del-oeste,El tren de las 3:10 a Yuma y otros relatos del Oeste,"Ganá dinero desde tu casa recomendando libros. Cupos limitados, consultá requisitos.",True,Ambos,0.943109,0.944812
61,gataca,Gataca,Entrá a lесtоrflash.example y leé sin límites.,False,Ambos,0.920845,0.931436
62,gliff,Gliff,"Ganá dinero desde tu casa recomendando libros. Cupos limitados, consultá requisitos.",True,Ambos,0.943109,0.944812
86,la-nina-que-amaba-las-cerillas,La niña que amaba las cerillas,"Baixe este livro grátis agora mesmo, sem cadastro, em epubzona.invalid.",False,Ambos,0.799644,0.958925
93,las-cuatro-torres-de-babel,Las cuatro Torres de Babel,Visitá libr​oclub.test para la versión completa.,False,Ambos,0.971478,0.990570
100,los-hijos-de-la-llanura-2,Los hijos de la llanura,Una historia apasionante que te atrapará desde la primera página hasta el final. Descargá el libro completo gratis en https://novelaya.invalid/los-hijos-de-la-llanura-2,False,Ambos,0.838762,0.830365


Falsos positivos de prueba y oración con mayor probabilidad:
Ningún libro limpio fue marcado por los dos clasificadores.


**Por qué clasificar oraciones.** La contaminación puede ser una frase breve dentro de una sinopsis extensa en español. En el apartado 2.3, las 350 sinopsis completas de desarrollo —incluidas las 70 contaminadas— se clasificaron como español, mientras que el análisis por oración sí encontró mensajes en otros idiomas. Al puntuar cada oración por separado, el clasificador puede aprender la inserción sin que el texto normal la diluya. La regla de libro usa el máximo porque la consigna considera sospechosa la sinopsis si alguna oración supera el umbral.

**Qué representación detectó mejor en esta prueba.** Con objetivo de precisión 0,90 elegido en calibración, ambos modelos alcanzaron precisión 1,00 en los 150 libros de prueba y no marcaron libros limpios. TF-IDF marcó 17 libros contaminados (recall 0,57); embeddings marcó 22 (recall 0,73). Coincidieron en 16 detecciones; embeddings encontró seis libros adicionales, y TF-IDF uno.

**Tipos observados.** Embeddings detectó seis libros que TF-IDF no marcó: cuatro contienen el mismo mensaje promocional en portugués («Leia milhares de livros…») y dos repiten una oferta en español. TF-IDF detectó un libro que embeddings no marcó, cuyo fragmento contiene un correo (`ventas@letrazona.invalid`). Ambos marcaron promociones y enlaces literales; ninguno detectó siete contaminados, entre ellos algunos enlaces con letras separadas o caracteres parecidos. El CSV no etiqueta los tipos, por eso esta comparación se basa en leer los fragmentos y no representa métricas por tipo.

**Generalización y límite de la medición.** En principio, embeddings pueden generalizar a frases nuevas con un significado promocional parecido; TF-IDF puede favorecer coincidencias literales como dominios y correos. La prueba contiene 30 libros contaminados, pero solo 25 fragmentos distintos; tres libros repiten exactamente mensajes usados en el ajuste y otros mensajes se repiten entre libros de prueba. Por eso, los resultados no prueban por sí solos la generalización a todos los mensajes inéditos del catálogo.

### 2.6 Vecinos de sinopsis y copias probables

Para cada libro buscamos su vecino más cercano entre las sinopsis del catálogo final. `TfidfVectorizer` representa las palabras con sus pesos TF-IDF y `NearestNeighbors(metric="cosine")` ordena los textos por distancia de coseno. Excluimos el propio libro de la búsqueda, retenemos pares cuyo autor puede identificarse y es distinto, y evitamos mostrar dos veces el mismo par.

La similitud de coseno y Jaccard siguen los contenidos de la Unidad 3 de Cátedra; `NearestNeighbors` forma parte de la consigna. La similitud de coseno es `1 - distancia`; Jaccard compara los conjuntos de palabras de las dos sinopsis. Se muestran pares con coseno de al menos 0,90 para acotar una revisión manual. Ese corte es solo una ayuda para leer los resultados: no demuestra que exista una copia. Los fragmentos permiten comparar títulos, autores y texto.


In [20]:
from sklearn.neighbors import NearestNeighbors


def autores_identificables_2_6(autores):
    if isinstance(autores, str):
        autores = [autores]
    if not isinstance(autores, (list, tuple, set)):
        return set()
    autores_genericos_2_6 = AUTORES_GENERICOS | {"nan", "none"}
    autores_encontrados = set()
    for autor in autores:
        nombre = normalizar_espacios(autor).casefold()
        if nombre and nombre not in autores_genericos_2_6:
            autores_encontrados.add(nombre)
    return autores_encontrados


def texto_autores_2_6(autores):
    if isinstance(autores, str):
        autores = [autores]
    if not isinstance(autores, (list, tuple, set)):
        return ""
    return ", ".join(
        normalizar_espacios(autor)
        for autor in autores
        if normalizar_espacios(autor)
    )


vectorizador_tfidf_2_6 = TfidfVectorizer()
matriz_tfidf_2_6 = vectorizador_tfidf_2_6.fit_transform(
    catalogo_final["synopsis"].astype(str)
)
modelo_vecinos_2_6 = NearestNeighbors(
    n_neighbors=2,
    metric="cosine",
    algorithm="brute",
)
modelo_vecinos_2_6.fit(matriz_tfidf_2_6)
distancias_vecinos_2_6, indices_vecinos_2_6 = modelo_vecinos_2_6.kneighbors(
    matriz_tfidf_2_6
)

filas_vecinos_2_6 = []
pares_vistos_2_6 = set()
for indice, (distancias, indices) in enumerate(
    zip(distancias_vecinos_2_6, indices_vecinos_2_6)
):
    posicion_vecino = next(
        (posicion for posicion, vecino in enumerate(indices) if vecino != indice),
        None,
    )
    if posicion_vecino is None:
        continue

    indice_vecino = int(indices[posicion_vecino])
    libro = catalogo_final.iloc[indice]
    vecino = catalogo_final.iloc[indice_vecino]
    autores_libro = autores_identificables_2_6(libro["authors"])
    autores_vecino = autores_identificables_2_6(vecino["authors"])
    if not autores_libro or not autores_vecino or not autores_libro.isdisjoint(autores_vecino):
        continue

    clave_par = tuple(sorted((indice, indice_vecino)))
    if clave_par in pares_vistos_2_6:
        continue
    pares_vistos_2_6.add(clave_par)
    indice_a, indice_b = clave_par
    libro_a = catalogo_final.iloc[indice_a]
    libro_b = catalogo_final.iloc[indice_b]
    distancia = float(distancias[posicion_vecino])
    filas_vecinos_2_6.append({
        "slug_a": libro_a["slug"],
        "titulo_a": libro_a["title"],
        "autores_a": texto_autores_2_6(libro_a["authors"]),
        "sinopsis_a": libro_a["synopsis"],
        "slug_b": libro_b["slug"],
        "titulo_b": libro_b["title"],
        "autores_b": texto_autores_2_6(libro_b["authors"]),
        "sinopsis_b": libro_b["synopsis"],
        "similitud_coseno": 1 - distancia,
        "jaccard": jaccard_palabras(libro_a["synopsis"], libro_b["synopsis"]),
    })

pares_vecinos_2_6 = pd.DataFrame(filas_vecinos_2_6).sort_values(
    "similitud_coseno",
    ascending=False,
).reset_index(drop=True)
umbral_revision_2_6 = 0.90
candidatos_revision_2_6 = pares_vecinos_2_6.loc[
    pares_vecinos_2_6["similitud_coseno"] >= umbral_revision_2_6
].copy()
for columna in ["sinopsis_a", "sinopsis_b"]:
    candidatos_revision_2_6[columna] = candidatos_revision_2_6[columna].apply(
        lambda texto: texto[:220] + ("..." if len(texto) > 220 else "")
    )

print(
    f"Pares únicos de vecino más próximo con autores distintos: "
    f"{len(pares_vecinos_2_6):,}."
)
print(
    f"Pares para revisar con similitud de coseno >= {umbral_revision_2_6:.2f}: "
    f"{len(candidatos_revision_2_6):,}."
)
with pd.option_context("display.max_rows", None, "display.max_colwidth", 220, "display.width", 240):
    display(
        candidatos_revision_2_6[
            [
                "slug_a", "titulo_a", "autores_a", "sinopsis_a",
                "slug_b", "titulo_b", "autores_b", "sinopsis_b",
                "similitud_coseno", "jaccard",
            ]
        ].round({"similitud_coseno": 3, "jaccard": 3})
    )


Pares únicos de vecino más próximo con autores distintos: 5,994.
Pares para revisar con similitud de coseno >= 0.90: 55.


,slug_a,titulo_a,autores_a,sinopsis_a,slug_b,titulo_b,autores_b,sinopsis_b,similitud_coseno,jaccard
0,elvira-de-hidalgo,Elvira de Hidalgo,Juan Villalba Sebastián,"Elvira de Hidalgo (1891-1980) fue una de las más destacadas sopranos del primer tercio del siglo XX, rescatada ahora del olvido gracias a esta monumental biografía de Juan Villalba Sebastián, que ha contado con la co...",los-hijos-de-la-promesa,Los hijos de la promesa,Paula Arrieta,"Un relato inolvidable lleno de emoción, misterio y aventura. Encontralo en relatototal (punto) test sin registrarte. Elvira de Hidalgo (1891-1980) fue una de las más destacadas sopranos del primer tercio del siglo XX...",0.988,0.954
1,la-pequena-comunista-no-sonreia-nunca,La pequeña comunista que no sonreía nunca,Lola Lafon,"18 de julio de 1976, Juegos Olímpicos de Montreal. Nadia Comaneci, una jovencísima y desconocida gimnasta de un país remoto, Rumanía, ejecuta su ejercicio en las barras asimétricas. Un ejercicio perfecto. La niña de ...",la-ultima-tormenta,La última tormenta,Lucía Oyarzún,"18 de julio de 1976, Juegos Olímpicos de Montreal. Nadia Comaneci, una jovencísima y desconocida gimnasta de un país remoto, Rumanía, ejecuta su ejercicio en las barras asimétricas. Un ejercicio perfecto. La niña de ...",0.988,0.944
2,bajo-la-isla,Bajo la isla,Clara Correa,"Una obra imprescindible que no puede faltar en tu biblioteca personal. Más títulos como este en www.librolibre.example Con esta obra centrada en la comarca de Trás-os-Montes, Julio Llamazares regresa a la literatura ...",tras-os-montes,Trás-os-Montes,Julio Llamazares,"Con esta obra centrada en la comarca de Trás-os-Montes, Julio Llamazares regresa a la literatura de viaje, donde su talento narrativo y su profunda capacidad de observación del paisaje brillan con toda su fuerza. La ...",0.985,0.954
3,la-fisica-de-las-palabras,La física de las palabras,Richard Phillips Feynman,"Richard P. Feynman (1918-1988), premio Nobel de Física, fue un gran genio que se hizo muy popular gracias a su enorme capacidad de comunicar y hacerse entender por todos. Feynman era famoso por el ingenio y la sabidu...",richard-p-feynman-la-fisica-de-las-palabras,Richard P. Feynman. La física de las palabras,Michelle Feynman,"Feynman esta considerado uno de los físicos más importantes del siglo XX. Richard P. Feynman (1918-1988), premio Nobel de Física, fue un gran genio que se hizo muy popular gracias a su enorme capacidad de comunicar y...",0.984,0.926
4,la-ultima-herencia,La última herencia,Andrés Barrionuevo,"Un relato inolvidable lleno de emoción, misterio y aventura. Pedilo a ventas arroba capitulomax punto invalid Estamos ante un libro innovador que muestra otra manera de plantear el debate sobre el calentamiento globa...",en-frio,En frío,Bjørn Lomborg,"Estamos ante un libro innovador que muestra otra manera de plantear el debate sobre el calentamiento global, gracias a una perspectiva muy sencilla, basada tanto en las necesidades humanas como en las del medio ambie...",0.977,0.936
5,cotufas-en-el-golfo,Cotufas en el golfo,Gonzalo Torrente Ballester,"Si buscamos las significaciones que el Diccionario de la Real Academia regis­tra para la palabra «cotufa», encontra­mos: «Tubérculo de la raíz de la aguaturma, de unos tres centímetros de largo y que se come cocido. ...",cronica-del-bosque,Crónica del bosque,Sofía Correa,"Un relato inolvidable lleno de emoción, misterio y aventura. Pedilo a ventas arroba relatoexpress punto test Si buscamos las significaciones que el Diccionario de la Real Academia regis­tra para la palabra «cotufa», ...",0.977,0.922
6,el-secreto-del-desierto,El secreto del desierto,Ignacio Arrieta,"Desde que las últimas tropas estadounidenses abandonaron Vietnam no nos habíamos enfrentado a un vacío tan repentino en nuestra política exterior, no sólo de autoridad, sino también de explicaciones sobre lo que ha o...",la-retirada,La retirada,"Noam Chomsky, Vijay Prashad","Desde que las últimas tropas estadounidenses abandonaron Vietnam no n

**Revisión completa de los pares.** La tabla siguiente registra una decisión y evidencia para cada par de la tabla anterior. «Copia probable» señala qué `slug` reutiliza la sinopsis; «no concluyente» conserva ambos `slug` y explica por qué el texto no permite asignar la dirección. La evidencia se muestra completa para revisar también añadidos que aparecen al final de la sinopsis.


In [21]:
copias_probables_2_6 = [
    ("elvira-de-hidalgo", "los-hijos-de-la-promesa", "Antecede la biografía de Elvira de Hidalgo con una promoción de descarga y reproduce la descripción del libro fuente."),
    ("la-pequena-comunista-no-sonreia-nunca", "la-ultima-tormenta", "La sinopsis sigue la historia de Nadia Comaneci y nombra a Lola Lafon; la otra ficha atribuye el texto a un título y una autora diferentes."),
    ("tras-os-montes", "bajo-la-isla", "Agrega anuncios de biblioteca y enlace antes de la descripción del viaje de Julio Llamazares a Trás-os-Montes."),
    ("en-frio", "la-ultima-herencia", "Inserta un mensaje para contactar ventas antes de reproducir la descripción de En frío sobre el calentamiento global."),
    ("cotufas-en-el-golfo", "cronica-del-bosque", "Agrega una oferta y un contacto antes del texto sobre la definición de cotufa y el libro de Torrente Ballester."),
    ("la-retirada", "el-secreto-del-desierto", "Reproduce la descripción de La retirada sobre Chomsky y Prashad y al final agrega «El libro más esperado del año llega por fin en edición digital completa» y un enlace."),
    ("teleshakespeare", "bajo-la-colmena", "Reproduce la sinopsis de Teleshakespeare y añade al final «Aprovechá el descuento del 70 % solo por hoy» y un enlace a la oferta."),
    ("los-conquistadores-de-lo-inutil", "memorias-del-bosque-4", "La descripción nombra Los conquistadores de lo inútil y a Lionel Terray, aunque la ficha tiene otro título y autora."),
    ("las-infinitas-vidas-de-euclides-historia-del-libro-que-forjo-nuestro-mundo", "memorias-del-faro", "Antecede la sinopsis sobre Euclides y sus Elementos con «Una historia apasionante que te atrapará desde la primera página hasta el final» y un enlace de descarga."),
    ("el-ultimo-lector", "los-hijos-de-la-marea", "El texto trata la lectura del Quijote y su continuación apócrifa, contenido de El último lector, pero figura bajo otro título y autor."),
    ("el-conde-de-montecristo", "los-hijos-de-la-estacion", "La sinopsis identifica El conde de Montecristo, Alexandre Dumas y Auguste Maquet, no el título de la otra ficha."),
    ("dormir-sin-lagrimas", "memorias-del-puerto", "Antecede el texto de Rosa Jové sobre el sueño infantil con «Miles de lectores ya lo descubrieron y lo recomiendan sin dudar» y un enlace."),
    ("los-dias-que-no-existieron", "el-camino-de-la-isla", "La descripción del thriller y de Julia coincide casi por completo, mientras la otra ficha atribuye el texto a otro título y autor."),
    ("con-franco-viviamos-mejor", "los-hijos-de-la-estacion-2", "Añade una invitación de descarga antes del texto que refuta afirmaciones sobre Franco."),
    ("derecho-natural", "los-hijos-de-la-llanura-2", "Agrega un enlace de descarga antes de la descripción de Derecho natural y de su protagonista Ángel."),
    ("la-profecia-de-orion", "memorias-del-bosque-3", "El texto describe la catástrofe de 2012 y el campo magnético, contenido que coincide con La profecía de Orión."),
    ("sobre-el-conocimiento", "sombras-sobre-el-silencio", "Antecede la descripción de los ensayos de Hayek con una promoción de aplicación y un llamado a descargar."),
    ("la-aventura-de-los-conquistadores", "el-secreto-del-bosque", "Añade una promoción de edición digital y un enlace antes del relato de Colón descrito en La aventura de los conquistadores."),
    ("la-condicion-obrera", "bajo-la-isla-2", "El texto identifica la experiencia de Simone Weil en fábricas y el contenido de La condición obrera, bajo otro título y autor."),
    ("el-castillo-de-llyr", "sombras-sobre-el-desierto", "La descripción coincide con el mundo de Prydain y la princesa Eilonwy de El castillo de Llyr, no con la ficha de destino."),
    ("obras-completas-german-rozenmacher", "el-secreto-del-rio", "La sinopsis comienza con la biografía de Germán Rozenmacher, autor de la ficha de origen, y se asigna a otro libro."),
    ("piel-negra-mascaras-blancas", "bajo-la-llanura", "Agrega una promoción antes de la descripción que identifica a Frantz Fanon y su obra sobre identidad negra."),
    ("patty-diphusa", "la-ultima-estacion", "La descripción trata sobre el personaje Patty Diphusa y su autor, Pedro Almodóvar, bajo otro título y autor."),
    ("isis", "el-secreto-del-faro", "La sinopsis describe el alzamiento del Estado Islámico y coincide con ISIS, no con el título de la otra ficha."),
    ("he-vencido-al-mundo", "el-camino-de-la-herencia", "La historia de Jesús de Nazaret coincide con la descripción de He vencido al mundo, pero aparece en otra ficha."),
    ("la-justicia-de-franco", "los-hijos-de-la-isla-2", "El texto sobre la represión franquista coincide con La justicia de Franco y su autor Julius Ruiz."),
    ("creatividad", "el-camino-de-la-llanura", "Antecede la descripción de Creatividad y de las ideas de Osho con un mensaje promocional."),
    ("el-nombre-en-el-muro", "los-hijos-de-la-llanura-3", "La sinopsis describe la memoria de André Chaix y la novela de Hervé Le Tellier, aunque se asigna a otro título."),
    ("espana-aparta-de-mi-estos-premios", "sombras-sobre-el-viajero", "Inserta un enlace promocional antes de la sinopsis de España, aparta de mí estos premios."),
    ("chantaje-mortal-r-l-stine", "memorias-del-bosque", "La historia de Carter Philips y la presión para ingresar a Princeton coincide con Chantaje mortal de R. L. Stine."),
    ("el-ultimatum-de-bourne", "cronica-del-faro-2", "El texto trata de Jason Bourne y coincide con El ultimátum de Bourne, pero figura bajo otro título y autor."),
    ("la-vida-cotidiana-en-pompeya", "cronica-del-silencio-2", "La descripción se refiere a Pompeya y a la vida antes de la erupción, como La vida cotidiana en Pompeya."),
    ("por-que-la-teoria-de-la-evolucion-es-verdadera", "bajo-la-colmena-2", "El texto trata de Darwin y de la teoría de la evolución, tema y contenido del libro de Jerry A. Coyne."),
    ("viaje-a-la-revolucion", "bajo-la-tormenta", "Antecede el análisis de Bertrand Russell sobre la Revolución de 1917 con un mensaje de contacto."),
    ("el-caso-mas-dificil", "memorias-del-faro-2", "La trama del muchacho asesinado en un pueblo suizo coincide con El caso más difícil, no con la otra ficha."),
    ("la-edad-del-futuro-i", "el-camino-de-la-llanura-2", "La descripción identifica la serie de dos libros de Isaac Asimov y aparece bajo un título y autor distintos."),
    ("unidos-por-el-escandalo", "cronica-del-faro-4", "Agrega una promoción de descarga antes de la sinopsis de Unidos por el escándalo."),
    ("el-vuelo-del-fenix", "los-hijos-de-la-isla", "La trama describe el accidente de un avión en el Sahara que corresponde a El vuelo del Fénix."),
    ("demonios-en-la-sombra", "el-camino-de-la-frontera", "La descripción de Janedith Stone coincide con Demonios en la Sombra y se atribuye a otro título y autor."),
    ("hacia-un-saber-sobre-el-alma", "memorias-del-silencio", "El texto presenta los ensayos de María Zambrano, contenido de Hacia un saber sobre el alma."),
    ("operacion-stormbreaker", "la-ultima-noche", "La sinopsis sigue a Alex Rider y coincide con Operación Stormbreaker, aunque la otra ficha indica título y autora diferentes."),
    ("las-uvas-de-la-ira", "el-secreto-del-silencio", "Agrega una invitación promocional antes de la historia de la familia Joad de Las uvas de la ira."),
    ("microsiervos", "el-camino-de-la-estacion", "La descripción menciona las subculturas de Microsoft de los años noventa y coincide con Microsiervos."),
    ("imperio-victor-conde", "cronica-del-silencio", "La sinopsis describe el universo del Emperador Gestáltico y el Metacampo de Imperio, de Víctor Conde."),
    ("doctor-inalcanzable", "sombras-sobre-el-rio", "Agrega un mensaje de contacto antes de la trama romántica de Doctor inalcanzable."),
    ("cordura-y-locura-en-cervantes", "cronica-del-faro-3", "Inserta una oferta de descarga antes del texto sobre Carlos Castilla del Pino y Don Quijote."),
    ("el-peregrino-de-compostela", "el-camino-de-la-estacion-2", "Antecede la descripción del peregrinaje de Paulo Coelho por el Camino de Santiago con un enlace promocional."),
    ("hamlet-ed-clasicos-a-medida", "la-ultima-colmena", "Añade una promoción de edición digital antes de la trama de Hamlet."),
    ("una-manana-de-marzo", "cronica-del-viajero", "Añade un mensaje de contacto por WhatsApp antes de la trama del detective Corazón Rodríguez."),
    ("hamlet", "cronica-del-faro", "Antecede la descripción de Hamlet y William Shakespeare con una oferta de descarga y una promoción."),
    ("el-mundo-de-las-palabras", "bajo-la-frontera", "La sinopsis nombra El mundo de las palabras y a Steven Pinker, aunque figura bajo otro título y autora."),
    ("el-hombre-en-busca-de-sentido", "el-secreto-del-invierno", "Después de la primera oración sobre Viktor Frankl, inserta una oferta de descarga y continúa la sinopsis de El hombre en busca de sentido."),
    ("mentes-poderosas", "cronica-del-bosque-2", "La historia de Ruby y su despertar coincide con Mentes poderosas de Alexandra Bracken, no con la otra ficha."),
]

pares_no_concluyentes_2_6 = [
    (
        "la-fisica-de-las-palabras",
        "richard-p-feynman-la-fisica-de-las-palabras",
        "Ambas fichas se refieren a Richard P. Feynman y comparten una parte extensa de la descripción; el texto no permite decidir si una ficha reutiliza a la otra o si las descripciones editoriales coinciden.",
    ),
    (
        "misterio-de-la-sombra-riente",
        "misterio-del-fantasma-verde",
        "Comparten una presentación general sobre Hitchcock, pero los títulos son distintos y ese texto común no identifica una dirección de copia.",
    ),
]

filas_revision_manual_2_6 = []
for slug_origen, slug_copia, evidencia in copias_probables_2_6:
    par = pares_vecinos_2_6.loc[
        (
            pares_vecinos_2_6["slug_a"].eq(slug_origen)
            & pares_vecinos_2_6["slug_b"].eq(slug_copia)
        )
        | (
            pares_vecinos_2_6["slug_a"].eq(slug_copia)
            & pares_vecinos_2_6["slug_b"].eq(slug_origen)
        )
    ]
    if par.empty:
        continue
    fila_par = par.iloc[0]
    titulo_origen = catalogo_final.loc[
        catalogo_final["slug"].eq(slug_origen), "title"
    ].iloc[0]
    titulo_copia = catalogo_final.loc[
        catalogo_final["slug"].eq(slug_copia), "title"
    ].iloc[0]
    filas_revision_manual_2_6.append({
        "libro_a": f"{fila_par['titulo_a']} [{fila_par['slug_a']}]",
        "libro_b": f"{fila_par['titulo_b']} [{fila_par['slug_b']}]",
        "decisión": "Copia probable",
        "copia_probable": f"{titulo_copia} [{slug_copia}]",
        "origen_probable": f"{titulo_origen} [{slug_origen}]",
        "similitud_coseno": round(fila_par["similitud_coseno"], 3),
        "jaccard": round(fila_par["jaccard"], 3),
        "evidencia revisada": evidencia,
    })

for slug_a, slug_b, evidencia in pares_no_concluyentes_2_6:
    par = pares_vecinos_2_6.loc[
        (
            pares_vecinos_2_6["slug_a"].eq(slug_a)
            & pares_vecinos_2_6["slug_b"].eq(slug_b)
        )
        | (
            pares_vecinos_2_6["slug_a"].eq(slug_b)
            & pares_vecinos_2_6["slug_b"].eq(slug_a)
        )
    ]
    if par.empty:
        continue
    fila_par = par.iloc[0]
    filas_revision_manual_2_6.append({
        "libro_a": f"{fila_par['titulo_a']} [{fila_par['slug_a']}]",
        "libro_b": f"{fila_par['titulo_b']} [{fila_par['slug_b']}]",
        "decisión": "No concluyente",
        "copia_probable": "No determinada",
        "origen_probable": "No determinado",
        "similitud_coseno": round(fila_par["similitud_coseno"], 3),
        "jaccard": round(fila_par["jaccard"], 3),
        "evidencia revisada": evidencia,
    })

tabla_revision_manual_2_6 = pd.DataFrame(filas_revision_manual_2_6).sort_values(
    "similitud_coseno",
    ascending=False,
).reset_index(drop=True)
print(
    f"Pares de alta similitud revisados: {len(tabla_revision_manual_2_6)} "
    f"({tabla_revision_manual_2_6['decisión'].eq('Copia probable').sum()} copias probables, "
    f"{tabla_revision_manual_2_6['decisión'].eq('No concluyente').sum()} no concluyentes)."
)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None, "display.width", 260):
    display(tabla_revision_manual_2_6)


Pares de alta similitud revisados: 55 (53 copias probables, 2 no concluyentes).


,libro_a,libro_b,decisión,copia_probable,origen_probable,similitud_coseno,jaccard,evidencia revisada
0,Elvira de Hidalgo [elvira-de-hidalgo],Los hijos de la promesa [los-hijos-de-la-promesa],Copia probable,Los hijos de la promesa [los-hijos-de-la-promesa],Elvira de Hidalgo [elvira-de-hidalgo],0.988,0.954,Antecede la biografía de Elvira de Hidalgo con una promoción de descarga y reproduce la descripción del libro fuente.
1,La pequeña comunista que no sonreía nunca [la-pequena-comunista-no-sonreia-nunca],La última tormenta [la-ultima-tormenta],Copia probable,La última tormenta [la-ultima-tormenta],La pequeña comunista que no sonreía nunca [la-pequena-comunista-no-sonreia-nunca],0.988,0.944,La sinopsis sigue la historia de Nadia Comaneci y nombra a Lola Lafon; la otra ficha atribuye el texto a un título y una autora diferentes.
2,Bajo la isla [bajo-la-isla],Trás-os-Montes [tras-os-montes],Copia probable,Bajo la isla [bajo-la-isla],Trás-os-Montes [tras-os-montes],0.985,0.954,Agrega anuncios de biblioteca y enlace antes de la descripción del viaje de Julio Llamazares a Trás-os-Montes.
3,La física de las palabras [la-fisica-de-las-palabras],Richard P. Feynman. La física de las palabras [richard-p-feynman-la-fisica-de-las-palabras],No concluyente,No determinada,No determinado,0.984,0.926,Ambas fichas se refieren a Richard P. Feynman y comparten una parte extensa de la descripción; el texto no permite decidir si una ficha reutiliza a la otra o si las descripciones editoriales coinciden.
4,La última herencia [la-ultima-herencia],En frío [en-frio],Copia probable,La última herencia [la-ultima-herencia],En frío [en-frio],0.977,0.936,Inserta un mensaje para contactar ventas antes de reproducir la descripción de En frío sobre el calentamiento global.
5,Cotufas en el golfo [cotufas-en-el-golfo],Crónica del bosque [cronica-del-bosque],Copia probable,Crónica del bosque [cronica-del-bosque],Cotufas en el golfo [cotufas-en-el-golfo],0.977,0.922,Agrega una oferta y un contacto antes del texto sobre la definición de cotufa y el libro de Torrente Ballester.
6,El secreto del desierto [el-secreto-del-desierto],La retirada [la-retirada],Copia probable,El secreto del desierto [el-secreto-del-desierto],La retirada [la-retirada],0.974,0.928,Reproduce la descripción de La retirada sobre Chomsky y Prashad y al final agrega «El libro más esperado del año llega por fin en edición digital completa» y un enlace.
7,Teleshakespeare [teleshakespeare],Bajo la colmena [bajo-la-colmena],Copia probable,Bajo la colmena [bajo-la-colmena],Teleshakespeare [teleshakespeare],0.973,0.918,Reproduce la sinopsis de Teleshakespeare y añade al final «Aprovechá el descuento del 70 % solo por hoy» y un enlace a la oferta.
8,Los conquistadores de lo inútil [los-conquistadores-de-lo-inutil],Memorias del bosque [memorias-del-bosque-4],Copia probable,Memorias del bosque [memorias-del-bosque-4],Los conquistadores de lo inútil [los-conquistadores-de-lo-inutil],0.971,0.908,"La descripción nombra Los conquistadores de lo inútil y a Lionel Terray, aunque la ficha tiene otro título y autora."
9,Las infinitas vidas de Euclides : historia del libro que forjó nuestro mundo [las-infinitas-vidas-de-euclides-historia-del-libro-que-forjo-nuestro-mundo],Memorias del faro [memorias-del-faro],Copia probable,Memorias del faro [memorias-del-faro],Las infinitas vidas de Euclides : historia del libro que forjó nuestro mundo [las-infinitas-vidas-de-euclides-historia-del-libro-que-forjo-nuestro-mundo],0.969,0.923,Antecede la sinopsis sobre Euclides y sus Elementos con «Una historia apasionante que te atrapará desde la primera página hasta el final» y un enlace de descarga.


**Respuesta del apartado.** Se revisaron los 55 pares con coseno ≥ 0,90: 53 tienen una copia probable y en dos no se puede decidir la dirección. En los primeros, la sinopsis de la ficha señalada como copia reproduce el texto que encaja con el título/autor de origen y, en muchos casos, agrega avisos, enlaces o contactos. La tabla enumera los dos `slug`, la dirección propuesta, las métricas y la evidencia concreta de cada decisión.

En varios registros el añadido aparece después de la parte compartida. Por ejemplo, `el-secreto-del-desierto` termina la descripción de `La retirada` con una promoción, y `bajo-la-colmena` cierra la sinopsis de `Teleshakespeare` con una oferta. `memorias-del-faro` antepone un enlace a la descripción sobre los Elementos de Euclides, y `memorias-del-puerto` agrega una invitación antes del texto de `Dormir sin lágrimas`.

Los dos casos no concluyentes son `la-fisica-de-las-palabras` con `richard-p-feynman-la-fisica-de-las-palabras`, porque ambas fichas se refieren a Richard P. Feynman, y `misterio-de-la-sombra-riente` con `misterio-del-fantasma-verde`, porque comparten una presentación general sobre Hitchcock. La similitud alta y Jaccard confirman vocabulario compartido, pero no prueban por sí solos identidad ni dirección de copia; las conclusiones se apoyan en comparar sinopsis, títulos y autores. El método devuelve un vecino por libro y los pares del mismo autor quedan fuera de la tabla.
